# 📉📈 <span style="color: white; background-color: DodgerBlue"><b> Projeto de Análise de Headcount: Ativos, Demitidos e Admitidos </b></span></p>

🧩 <span style="color: DeepSkyBlue"><b> 1- Estruturação do Modelo de Dados </b></span></p>
- O projeto consolida três bases principais:
    - Ativos – colaboradores em atividade  
    - Admitidos – novos ingressos  
    - Demitidos – desligamentos no período  
- Cada base passa por:
    - Normalização de colunas  
    - Padronização de datas  
    - Criação de chaves de relacionamento  
    - Correção de inconsistências  
    - Tratamento de valores nulos  
- Essas bases são integradas em um modelo único no HTML

🧮 <span style="color: DeepSkyBlue"><b> 2- Métricas e KPIs calculados </b></span></p>
- O painel produz indicadores essenciais para gestão de pessoas, tais como:
    - Headcount Total - Número total de colaboradores ativos
    - Admitidos no período - Quantidade (absoluta) de novas contratações
    - Demitidos no período - Total de desligamentos registrados
    - Saldo de Headcount - HC Final = HC Inicial + Admitidos – Demitidos
    - Evolução Temporal 
- Comportamento do quadro por:
    - mês  
    - unidade  
    - diretoria  
    - cargo  
    - centro de custo  
- Análises adicionais:
    - Distribuição por gênero  
    - Distribuição por faixa etária  
    - Distribuição por tipo de contrato  
    - Movimentação interna  
    - Tempo médio de empresa  
    - Motivos de desligamento  

🧼 <span style="color: DeepSkyBlue"><b> 3- Tratamento e limpeza das bases </b></span></p>
- Conversão de datas para formato padrão  
- Categorização de colaboradores  
- Ajuste de vínculos e duplicidades  
- Padronização dos centros de custo  
- Integração com tabelas auxiliares (ex.: De-Para de Cargos ou Unidades)  
- Isso garante consistência entre diferentes fontes de informação

📊 <span style="color: DeepSkyBlue"><b> 4- Construção do painel no HTML </b></span></p>
- Estrutura típica:
    - Cards com KPIs principais  
    - Gráficos de barras para admitidos/demitidos  
    - Linha temporal do headcount  
    - Gráficos de pizza/donut para distribuição (sexo, faixa etária, etc.)  
    - Tabelas detalhadas e funções de drill-through  
- Padrões visuais adotados:
    - Paleta corporativa  
    - Layout limpo e organizado  
    - Foco em leitura executiva  
    - Hierarquias e segmentadores por diretoria, unidade e centro de custo  

 📧 <span style="color: DeepSkyBlue"><b> 5- Envio de e-mail à equipe de Gestão de Pessoas </b></span></p>
- Encaminha e-mail com o HTML em anexo contendo as informações no corpo do e-mail: 
    - Headcount Ativo
    - Demissões (Inativos)
    - Admissões
    - Turnover Acumulado
    - Demissões < 90 dias

🧾 <span style="color: DeepSkyBlue"><b> 6- Registro de execução </b></span></p>
- Registro de Etapas no arquivo PROCESSOS.xlsx  
- Log detalhado por execução  
- Auditoria completa do pipeline 

In [1]:
# Importando as Bibliotecas

import base64
import gc
import logging
import numpy as np
import os
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import pythoncom
import re
import sys
import threading
import traceback
import webbrowser
import win32com.client as win32
from datetime import datetime
from dotenv import load_dotenv
from http.server import HTTPServer, SimpleHTTPRequestHandler
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl import Workbook
from openpyxl import load_workbook
from openpyxl.worksheet.table import Table, TableStyleInfo
from plotly.subplots import make_subplots
from typing import List

# Carregamento da base de controle de processos

id = 22

path_registros_processos = r'X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\PROCESSOS.xlsx'

registros_processos = pd.read_excel(path_registros_processos, sheet_name = "REGISTROS", engine='openpyxl')

wb_p = load_workbook(path_registros_processos)

ws_p = wb_p['REGISTROS']

# Controle de atualização de processo: Etapa 1

tempo_0 = [id, datetime.today(), 0]

ws_p.append(tempo_0)

wb_p.save(path_registros_processos)

# ==============================================================================
# Configurar logging
# ==============================================================================

logging.basicConfig(
    level=logging.INFO,
    format='[%(asctime)s] [%(levelname)s] %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)

print("✓ Bibliotecas importadas com sucesso.\n")

# ==============================================================================
# FUNÇÃO DE LIMPEZA DE DADOS
# ==============================================================================

# Limpa valores inválidos do DataFrame
def limpar_dados_completo(df):
    for coluna in df.columns:
        try:
            if df[coluna].dtype == 'object':
                df[coluna] = df[coluna].apply(lambda x: 
                    np.nan if pd.isna(x) or (isinstance(x, str) and x.strip().upper() in ['NAN', 'NONE', '', 'NULL', 'N/A']) 
                    else x
                )
            elif df[coluna].dtype in ['float64', 'int64']:
                df[coluna] = pd.to_numeric(df[coluna], errors='coerce')
        except Exception as e:
            continue
    return df

# ==============================================================================
# FUNÇÃO UNIFICADA DE FORMATAÇÃO EXCEL
# ==============================================================================

# Formata arquivo Excel com cabeçalho azul e regras específicas de colunas.
def formatar_excel_profissional(caminho_arquivo, cor_cabecalho='005A9C', cor_fonte_cabecalho='FFFFFF'):
    try:
        wb = load_workbook(caminho_arquivo)
        
        for ws in wb.sheetnames:
            sheet = wb[ws]
            
            # ===== 1. MAPEAMENTO DE COLUNAS =====
            # Cria um dicionário para aplicar regras por nome
            col_map = {}
            for cell in sheet[1]:
                col_map[cell.column] = str(cell.value).lower().strip() if cell.value else ""

            # ===== 2. FORMATAR CABEÇALHO =====
            header_fill = PatternFill(start_color=cor_cabecalho, end_color=cor_cabecalho, fill_type='solid')
            header_font = Font(bold=True, color=cor_fonte_cabecalho, size=11)
            header_alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
            
            header_border = Border(
                left=Side(style='thin', color='FFFFFF'),
                right=Side(style='thin', color='FFFFFF'),
                top=Side(style='thin', color='FFFFFF'),
                bottom=Side(style='thin', color='FFFFFF')
            )

            for cell in sheet[1]:
                cell.fill = header_fill
                cell.font = header_font
                cell.alignment = header_alignment
                cell.border = header_border
            
            # ===== 3. FORMATAR DADOS =====
            data_fill = PatternFill(start_color='FFFFFF', end_color='FFFFFF', fill_type='solid')
            data_font = Font(size=10, color='333333')
            data_alignment = Alignment(horizontal='left', vertical='center', wrap_text=True)
            
            data_border = Border(
                left=Side(style='thin', color='DDDDDD'),
                right=Side(style='thin', color='DDDDDD'),
                top=Side(style='thin', color='DDDDDD'),
                bottom=Side(style='thin', color='DDDDDD')
            )

            # Colunas que devem ser TEXTO
            cols_texto = ['registro', 'nome', 'rg', 'cpf']
            
            # Colunas que devem ser INTEIRO (sem casas decimais)
            cols_inteiro = ['dias_trabalhado']

            for row in sheet.iter_rows(min_row=2, max_row=sheet.max_row, min_col=1, max_col=sheet.max_column):
                for cell in row:
                    cell.fill = data_fill
                    cell.font = data_font
                    cell.alignment = data_alignment
                    cell.border = data_border
                    
                    # Identifica o nome da coluna atual
                    col_name = col_map.get(cell.column, "")
                    
                    # --- REGRA 1: TEXTO ---
                    if col_name in cols_texto:
                        cell.number_format = '@'  # Formato Texto no Excel
                    
                    # --- REGRA 2: INTEIRO ---
                    elif col_name in cols_inteiro:
                        cell.number_format = '0'  # Inteiro
                        # Para separador de milhar sem decimais (ex: 1.200), usar '#,##0'
                    
                    # --- REGRA 3: DATAS ---
                    elif col_name in ['nascimento', 'data_admissao', 'data_rescisao', 'data_nasc_conjuge']:
                        cell.number_format = 'dd/mm/yyyy'
                    
                    # --- REGRA 4: OUTROS NÚMEROS ---
                    elif isinstance(cell.value, (int, float)) and not isinstance(cell.value, bool):
                        if isinstance(cell.value, float) or (isinstance(cell.value, int) and cell.value > 999):
                            cell.number_format = '#,##0.00'
                        else:
                            cell.number_format = '#,##0'
            
            # ===== 4. AJUSTAR LARGURA DAS COLUNAS =====
            for col_idx, col_name in enumerate(sheet.columns, 1):
                max_length = 0
                column_letter = get_column_letter(col_idx)
                
                header_cell = sheet[f'{column_letter}1']
                if header_cell.value:
                    max_length = len(str(header_cell.value))
                
                for row_idx in range(2, min(sheet.max_row + 1, 101)):
                    cell = sheet[f'{column_letter}{row_idx}']
                    if cell.value:
                        max_length = max(max_length, len(str(cell.value)))
                
                adjusted_width = min(max(max_length + 2, 12), 50)
                sheet.column_dimensions[column_letter].width = adjusted_width
            
            sheet.freeze_panes = 'A2'
            logging.info(f"   ✓ Aba '{ws}' formatada com sucesso")
        
        wb.save(caminho_arquivo)
        logging.info(f"✅ Arquivo '{caminho_arquivo}' formatado e salvo!\n")
        return True
        
    except Exception as e:
        logging.error(f"❌ Erro ao formatar Excel '{caminho_arquivo}': {e}\n")
        return False

# ==============================================================================
# 1. CARREGAMENTO E PRÉ-PROCESSAMENTO
# ==============================================================================

logging.info("="*80)
logging.info("1. CARREGAMENTO E PRÉ-PROCESSAMENTO DOS DADOS")
logging.info("="*80)

try:
    path_arquivo = r'X:\Gestão de Pessoas\Analytics\10 - Relatórios\10.4 - HC e Atestados Médicos\Controle_HC e Atestados.xlsx'
    dados_hc = pd.read_excel(path_arquivo, sheet_name='HC', engine='openpyxl')
    logging.info(f"✓ Base de dados carregada com {dados_hc.shape[0]} registros e {dados_hc.shape[1]} colunas.")

    cols_texto = ['registro', 'nome', 'rg', 'cpf']
    
    for col in cols_texto:
        if col in dados_hc.columns:

            dados_hc[col] = dados_hc[col].astype(str).str.replace(r'\.0$', '',  regex=True).str.strip()
            
    if 'dias_trabalhado' in dados_hc.columns:
        dados_hc['dias_trabalhado'] = pd.to_numeric(dados_hc['dias_trabalhado'], errors='coerce').fillna(0).astype(int)

    # Limpar dados
    dados_hc = limpar_dados_completo(dados_hc)

    # Converter colunas de data

    colunas_data = ['nascimento', 'data_admissao', 'data_rescisao', 'data_nasc_conjuge']
    data_base = pd.Timestamp('1899-12-30')

    for coluna in colunas_data:
        if coluna in dados_hc.columns:
            try:
                # CASO 1: Se o Pandas já leu como data nativa (datetime), apenas formata para BR
                if pd.api.types.is_datetime64_any_dtype(dados_hc[coluna]):
                    dados_hc[coluna] = dados_hc[coluna].dt.strftime('%d/%m/%Y')
                    
                else:
                    # Tenta converter para número para checar se é Número de Série do Excel (ex: 46181)
                    valores_numericos = pd.to_numeric(dados_hc[coluna], errors='coerce')
                        
                    # Se houver números válidos e eles estiverem no range padrão do Excel (1 a 100.000)
                    if not valores_numericos.dropna().empty and valores_numericos.dropna().between(1, 100000).all():
                        dados_hc[coluna] = data_base + pd.to_timedelta(valores_numericos, unit='D')
                        dados_hc[coluna] = dados_hc[coluna].dt.strftime('%d/%m/%Y')
                    else:
                        # CASO 3: Se for texto puro/string (ex: "08/06/2026")
                        dados_hc[coluna] = pd.to_datetime(dados_hc[coluna], errors='coerce', dayfirst=True).dt.strftime('%d/%m/%Y')
            except Exception as e:
                logging.warning(f"Aviso ao converter a coluna de data [{coluna}]: {e}")
                pass

    logging.info("✓ Colunas de data convertidas.")

    # Preencher valores faltantes
    dados_hc['filhos'] = dados_hc['filhos'].fillna('Não informado')
    dados_hc['secao'] = dados_hc['secao'].fillna('Não especificado')    
    dados_hc['centro_custo'] = dados_hc['centro_custo'].fillna('Não especificado')
    dados_hc['etnia_raca'] = dados_hc['etnia_raca'].fillna('Não informado')
    dados_hc['estado_civil'] = dados_hc['estado_civil'].fillna('Não informado')

    # Codificar logo
    caminho_imagem = 'Logo AFPESP.png'
    try:
        with open(caminho_imagem, "rb") as image_file:
            encoded_string = base64.b64encode(image_file.read()).decode('utf-8')
        img_src_base64 = f"data:image/png;base64,{encoded_string}"
    except:
        img_src_base64 = ""

    logging.info("✓ SEÇÃO 1 CONCLUÍDA COM SUCESSO\n")

except Exception as e:
    logging.error(f"ERRO na SEÇÃO 1: {e}")
    sys.exit("Script encerrado.")

# ==============================================================================
# 2. ANÁLISE DE COLABORADORES ATIVOS
# ==============================================================================

logging.info("="*80)
logging.info("2. ANÁLISE DE COLABORADORES ATIVOS")
logging.info("="*80)

try:
    hc_ativos = dados_hc.loc[dados_hc['descricao_rescisao'] == 'ATIVO'].copy()
    logging.info(f"✓ {hc_ativos.shape[0]} colaboradores ativos identificados.")

    # Estatísticas descritivas
    colunas_quant = ['salario_total', 'idade', 'dias_trabalhado', 'primeiro_atestado', 'dias_afastado', 'custo_afastamento', 'treinamentos']
    df_quant_ativo = hc_ativos[colunas_quant].copy()
    df_quant_ativo.columns = ['Salário', 'Idade', 'Dias Trabalhado', 'Primeiro Atestado', 'Dias Afastado', 'Custo Afastamento', 'Horas de Treinamentos']
    estat_ativo = df_quant_ativo.describe().round(2)

    # Card headcount
    headcount_ativo = hc_ativos.shape[0]
    fig_card_ativo = go.Figure(go.Indicator(
        mode="number",
        value=headcount_ativo,
        title={"text": "Headcount Ativo"},
        number={'font': {'size': 60, 'color': '#0070C0'}},
    ))
    fig_card_ativo.update_layout(height=150, margin=dict(l=0, r=0, t=30, b=0), paper_bgcolor="white")
    card_ativo_html = fig_card_ativo.to_html(full_html=False, include_plotlyjs='cdn')

    # Gráfico de Pizza - Sexo
    fig_sexo_ativo = px.pie(
        hc_ativos['sexo'].value_counts().reset_index(),
        values='count',
        names='sexo',
        title='Distribuição por Gênero',
        hole=0.3,
        color_discrete_sequence=['#0070C0', '#FF6B6B']
    )
    fig_sexo_ativo.update_layout(height=400, margin=dict(l=40, r=40, t=60, b=40))
    grafico_sexo_ativo = fig_sexo_ativo.to_html(full_html=False, include_plotlyjs=False)

    # Gráfico de Pizza - Filhos
    fig_filhos_ativo = px.pie(
        hc_ativos['filhos'].value_counts().reset_index(),
        values='count',
        names='filhos',
        title='Distribuição por Filhos',
        hole=0.3,
        color_discrete_sequence=['#0070C0', '#FF6B6B', '#4ECDC4']
    )
    fig_filhos_ativo.update_layout(height=400, margin=dict(l=40, r=40, t=60, b=40))
    grafico_filhos_ativo = fig_filhos_ativo.to_html(full_html=False, include_plotlyjs=False)

    # Gráfico de Barras - Etnia e Raça
    fig_etnia_ativo = px.bar(
        hc_ativos['etnia_raca'].value_counts().reset_index().sort_values('count'),
        x='count',
        y='etnia_raca',
        orientation='h',
        title='Distribuição por Etnia e Raça',
        color_discrete_sequence=['#0070C0']
    )
    fig_etnia_ativo.update_layout(
        height=400,
        margin=dict(l=120, r=40, t=60, b=40),
        showlegend=False,
        xaxis_title='Quantidade',
        yaxis_title='Etnia e Raça')
    grafico_etnia_ativo = fig_etnia_ativo.to_html(full_html=False, include_plotlyjs=False)

    # Gráfico de Barras - Estado Civil
    fig_civil_ativo = px.bar(
        hc_ativos['estado_civil'].value_counts().reset_index().sort_values('count'),
        x='count',
        y='estado_civil',
        orientation='h',
        title='Distribuição por Estado Civil',
        color_discrete_sequence=['#0070C0']
    )
    fig_civil_ativo.update_layout(
        height=400, 
        margin=dict(l=120, r=40, t=60, b=40), 
        showlegend=False,
        xaxis_title='Quantidade',
        yaxis_title='Estado Civil')
    grafico_civil_ativo = fig_civil_ativo.to_html(full_html=False, include_plotlyjs=False)

    # Gráfico de Barras - Formação
    fig_form_ativo = px.bar(
        hc_ativos['formacoes'].value_counts().reset_index().sort_values('count'),
        x='count',
        y='formacoes',
        orientation='h',
        title='Distribuição por Formação',
        color_discrete_sequence=['#0070C0']
    )
    fig_form_ativo.update_layout(
        height=500, 
        margin=dict(l=250, r=40, t=60, b=40), 
        showlegend=False, 
        xaxis_title='Quantidade',
        yaxis_title='Formação')
    grafico_form_ativo = fig_form_ativo.to_html(full_html=False, include_plotlyjs=False)

    # Gráfico de Barras - Cargo
    fig_cargo_ativo = px.bar(
        hc_ativos['cargo_completo'].value_counts().head(15).reset_index().sort_values('count'),
        x='count',
        y='cargo_completo',
        orientation='h',
        title='Top 15 Cargos',
        color_discrete_sequence=['#0070C0']
    )
    fig_cargo_ativo.update_layout(
        height=500, 
        margin=dict(l=200, r=40, t=60, b=40), 
        showlegend=False, 
        xaxis_title='Quantidade', 
        yaxis_title='Cargo')
    grafico_cargo_ativo = fig_cargo_ativo.to_html(full_html=False, include_plotlyjs=False)

    # Gráfico de Barras - Centro de Custo
    fig_cc_ativo = px.bar(
        hc_ativos['centro_custo'].value_counts().reset_index().sort_values('count'),
        x='count',
        y='centro_custo',
        orientation='h',
        title='Distribuição por Centro de Custo',
        color_discrete_sequence=['#0070C0']
    )
    fig_cc_ativo.update_layout(
        height=500, 
        margin=dict(l=200, r=40, t=60, b=40), 
        showlegend=False, 
        xaxis_title='Quantidade', 
        yaxis_title='Centro de Custo')
    grafico_cc_ativo = fig_cc_ativo.to_html(full_html=False, include_plotlyjs=False)

    # Gráfico de Barras - Unidade
    fig_unidade_ativo = px.bar(
        hc_ativos['empresa_resumo'].value_counts().reset_index().sort_values('count'),
        x='count',
        y='empresa_resumo',
        orientation='h',
        title='Distribuição por Unidade',
        color_discrete_sequence=['#0070C0']
    )
    fig_unidade_ativo.update_layout(
        height=600, 
        margin=dict(l=150, r=40, t=60, b=40), 
        showlegend=False, 
        xaxis_title='Quantidade', 
        yaxis_title='Empresa')
    grafico_unidade_ativo = fig_unidade_ativo.to_html(full_html=False, include_plotlyjs=False)

    # Boxplots
    fig_boxplot_ativo = make_subplots(
        rows=3, cols=2,
        subplot_titles=('Salário', 'Idade', 'Dias Trabalhado', 'Primeiro Atestado', 'Dias Afastado', 'Custo Afastamento'),
        specs=[[{'type': 'box'}, {'type': 'box'}], [{'type': 'box'}, {'type': 'box'}], [{'type': 'box'}, {'type': 'box'}]]
    )

    fig_boxplot_ativo.add_trace(go.Box(y=hc_ativos['salario_total'], name='Salário', marker_color='#0070C0'), row=1, col=1)
    fig_boxplot_ativo.add_trace(go.Box(y=hc_ativos['idade'], name='Idade', marker_color='#0070C0'), row=1, col=2)
    fig_boxplot_ativo.add_trace(go.Box(y=hc_ativos['dias_trabalhado'], name='Dias Trabalhado', marker_color='#0070C0'), row=2, col=1)
    fig_boxplot_ativo.add_trace(go.Box(y=hc_ativos['primeiro_atestado'], name='Primeiro Atestado', marker_color='#0070C0'), row=2, col=2)
    fig_boxplot_ativo.add_trace(go.Box(y=hc_ativos['dias_afastado'], name='Dias Afastado', marker_color='#0070C0'), row=3, col=1)
    fig_boxplot_ativo.add_trace(go.Box(y=hc_ativos['custo_afastamento'], name='Custo Afastamento', marker_color='#0070C0'), row=3, col=2)  

    fig_boxplot_ativo.update_layout(height=1000, showlegend=False, title_text="Boxplots - Variáveis Numéricas")
    boxplot_ativo_html = fig_boxplot_ativo.to_html(full_html=False, include_plotlyjs=False)

    # Tabela de estatísticas
    tabela_estat_ativo = estat_ativo.to_html(classes='tabela-estatistica', border=0)

    # Tabela de colaboradores
    colunas_tabela_ativos = ['registro', 'nome', 'idade', 'rg', 'cpf', 'data_admissao', 'situacao', 'dias_trabalhado',
                             'cargo_completo', 'salario_total', 'secao', 'centro_custo', 'empresa_resumo', 'formacoes']
    tabela_ativos = hc_ativos[colunas_tabela_ativos].to_html(index=False, classes='tabela-dados', border=0)

    logging.info("✓ SEÇÃO 2 CONCLUÍDA COM SUCESSO\n")

except Exception as e:
    logging.error(f"ERRO na SEÇÃO 2: {e}")
    traceback.print_exc()
    card_ativo_html = "Erro ao processar dados de ativos"
    grafico_sexo_ativo = ""
    grafico_filhos_ativo = ""
    grafico_etnia_ativo = ""
    grafico_civil_ativo = ""
    grafico_form_ativo = ""
    grafico_cargo_ativo = ""
    grafico_cc_ativo = ""
    grafico_unidade_ativo = ""
    boxplot_ativo_html = ""
    tabela_estat_ativo = ""
    tabela_ativos = ""

# ==============================================================================
# 3. ANÁLISE DE DEMISSÕES
# ==============================================================================

logging.info("="*80)
logging.info("3. ANÁLISE DE DEMISSÕES")
logging.info("="*80)

try:
    hc_inativos = dados_hc.loc[dados_hc['ano_rescisao'] == 2026].copy()
    logging.info(f"✓ {hc_inativos.shape[0]} demissões em 2026 identificadas.")

    # Card headcount
    headcount_inativo = hc_inativos.shape[0]
    fig_card_inativo = go.Figure(go.Indicator(
        mode="number",
        value=headcount_inativo,
        title={"text": "Demissões 2026"},
        number={'font': {'size': 60, 'color': '#FF6B6B'}},
    ))
    fig_card_inativo.update_layout(height=150, margin=dict(l=0, r=0, t=30, b=0), paper_bgcolor="white")
    card_inativo_html = fig_card_inativo.to_html(full_html=False, include_plotlyjs='cdn')

    # Gráfico de Corrida
    hc_inativos['Data de Rescisão'] = pd.to_datetime(hc_inativos['data_rescisao'], format='%d/%m/%Y', errors='coerce')
    hc_inativos_corrida = hc_inativos.dropna(subset=['Data de Rescisão'])

    if len(hc_inativos_corrida) > 0:
        contagem_corrida = hc_inativos_corrida.groupby(['empresa_resumo', 'Data de Rescisão']).size().reset_index(name='Demitidos')
        
        todas_empresas = contagem_corrida['empresa_resumo'].unique()
        todas_datas = pd.date_range(
            start=contagem_corrida['Data de Rescisão'].min(),
            end=contagem_corrida['Data de Rescisão'].max()
        )
        
        combinacoes = pd.MultiIndex.from_product(
            [todas_empresas, todas_datas],
            names=['empresa_resumo', 'Data de Rescisão']
        ).to_frame(index=False)
        
        contagem_completo = pd.merge(combinacoes, contagem_corrida, on=['empresa_resumo', 'Data de Rescisão'], how='left')
        contagem_completo['Demitidos'] = contagem_completo['Demitidos'].fillna(0)
        
        # 1. Ordenar e calcular o acumulado
        contagem_completo = contagem_completo.sort_values(by=['empresa_resumo', 'Data de Rescisão'])
        contagem_completo['Demissão Acumulado'] = contagem_completo.groupby('empresa_resumo')['Demitidos'].cumsum()
        contagem_completo = contagem_completo.rename(columns={'empresa_resumo': 'Empresa'})

        # 2. Calcular o Rank (posição) de cada empresa a cada dia
        # method='first' garante que não haverá empates na mesma posição do eixo Y
        contagem_completo['Rank'] = contagem_completo.groupby('Data de Rescisão')['Demissão Acumulado'].rank(method='first', ascending=True)

        # 3. Ordenar o dataframe pela Data e depois pelo Rank para a animação fluir corretamente
        contagem_completo = contagem_completo.sort_values(by=['Data de Rescisão', 'Rank'])

        # Formatar a data para o padrão BR
        contagem_completo['Data Formatada'] = contagem_completo['Data de Rescisão'].dt.strftime('%d/%m/%Y')
        
        # 4. Criar um texto combinando o nome da empresa e o valor para exibir na barra
        contagem_completo['Texto_Barra'] = contagem_completo['Empresa'] + ' - ' + contagem_completo['Demissão Acumulado'].astype(int).astype(str)

        # Gerar o gráfico
        fig_corrida_inativo = px.bar(
            contagem_completo,
            x='Demissão Acumulado',
            y='Rank', # O eixo Y é o Rank numérico, permitindo a troca de posições
            orientation='h',
            color='Empresa',
            text='Texto_Barra', # Mostra o nome da empresa e o valor dentro da barra
            animation_frame='Data Formatada',
            animation_group='Empresa',
            title='Demissões Acumuladas por Unidade'
        )
        
        fig_corrida_inativo.update_layout(
            height=600,
            margin=dict(l=40, r=40, t=60, b=40),
            yaxis=dict(
                showticklabels=False, # Oculta os números 1, 2, 3... do eixo Y
                title='',             # Remove o título do eixo Y
                range=[0.5, len(todas_empresas) + 0.5] # Mantém o eixo fixo para as barras não pularem
            ),
            xaxis_title='Demissões Acumuladas'
        )
        
        # Ajustar a posição do texto para ficar sempre visível (dentro ou fora da barra)
        fig_corrida_inativo.update_traces(textposition='outside')
        
        grafico_corrida_inativo = fig_corrida_inativo.to_html(full_html=False, include_plotlyjs=False)
    else:
        grafico_corrida_inativo = "<p>Sem dados para gráfico de corrida</p>"

    # Gráfico por Mês
    hc_inativos['mes_rescisao'] = pd.to_numeric(hc_inativos['mes_rescisao'], errors='coerce')
    meses_nome = {1: 'Jan', 2: 'Fev', 3: 'Mar', 4: 'Abr', 5: 'Mai', 6: 'Jun', 
                  7: 'Jul', 8: 'Ago', 9: 'Set', 10: 'Out', 11: 'Nov', 12: 'Dez'}

    contagem_mes = hc_inativos['mes_rescisao'].value_counts().sort_index()
    nomes_meses = contagem_mes.index.map(meses_nome)

    # --- AJUSTE PARA COMPARATIVO ANO CONTRA ANO (DEMISSÕES) ---
    df_yoy_dem = dados_hc.loc[dados_hc['ano_rescisao'].isin([2025, 2026])].copy()
    
    # Converter ano para string para o Plotly tratar como categoria
    df_yoy_dem['ano_rescisao'] = df_yoy_dem['ano_rescisao'].astype(int).astype(str)
    df_yoy_dem['mes_rescisao'] = pd.to_numeric(df_yoy_dem['mes_rescisao'], errors='coerce')
    
    contagem_yoy_dem = df_yoy_dem.groupby(['mes_rescisao', 'ano_rescisao']).size().reset_index(name='Quantidade')
    contagem_yoy_dem['Mês'] = contagem_yoy_dem['mes_rescisao'].map(meses_nome)
    contagem_yoy_dem = contagem_yoy_dem.sort_values('mes_rescisao')

    fig_mes_inativo = px.bar(
        contagem_yoy_dem,
        x='Mês',
        y='Quantidade',
        color='ano_rescisao', 
        barmode='group',  
        title='Demissões: 2025 vs 2026',
        color_discrete_map={'2025': '#D97373', '2026': '#DEB521'}
    )
    
    fig_mes_inativo.update_layout(
        height=400, 
        margin=dict(l=40, r=40, t=60, b=40), 
        legend_title_text='Ano',
        xaxis_title='Mês',
        yaxis_title='Quantidade')
    
    grafico_mes_inativo = fig_mes_inativo.to_html(full_html=False, include_plotlyjs=False)

    # Gráfico de Tipo de Rescisão
    # --- COMPARATIVO ANO CONTRA ANO (TIPO DE RESCISÃO) ---
    df_yoy_tipo = dados_hc.loc[dados_hc['ano_rescisao'].isin([2025, 2026])].copy()
    
    # Ano em texto para a legenda ficar limpa
    df_yoy_tipo['ano_rescisao'] = df_yoy_tipo['ano_rescisao'].astype(int).astype(str)
    
    # Agrupado por Tipo de Rescisão e Ano
    contagem_tipo = df_yoy_tipo.groupby(['descricao_rescisao', 'ano_rescisao']).size().reset_index(name='Quantidade')
    
    # Ordenação: os tipos com mais demissões apareçam primeiro
    ordem_tipos = contagem_tipo.groupby('descricao_rescisao')['Quantidade'].sum().sort_values(ascending=True).index

    fig_tipo_inativo = px.bar(
        contagem_tipo,
        x='Quantidade',
        y='descricao_rescisao',
        orientation='h',
        color='ano_rescisao',
        barmode='group', # Coloca as barras de ano lado a lado
        title='Distribuição por Tipo de Rescisão: 2025 vs 2026',
        color_discrete_map={'2025': '#D97373', '2026': '#DEB521'},
        category_orders={'descricao_rescisao': list(ordem_tipos)} # Mantém a ordem visual
    )
    
    fig_tipo_inativo.update_layout(
        height=500,
        margin=dict(l=250, r=40, t=60, b=40),
        legend_title_text='Ano',
        yaxis_title='Descrição da Rescisão',
        xaxis_title='Quantidade')
    
    grafico_tipo_inativo = fig_tipo_inativo.to_html(full_html=False, include_plotlyjs=False)

    # Tabela de demitidos
    colunas_tabela_inativos = ['registro', 'nome', 'idade', 'rg', 'cpf', 'data_admissao', 'data_rescisao', 'dias_trabalhado',
                               'descricao_rescisao', 'cargo_completo', 'salario_total', 'empresa_resumo', 'formacoes']
    tabela_inativos = hc_inativos[colunas_tabela_inativos].to_html(index=False, classes='tabela-dados', border=0)

    logging.info("✓ SEÇÃO 3 CONCLUÍDA COM SUCESSO\n")

except Exception as e:
    logging.error(f"ERRO na SEÇÃO 3: {e}")
    traceback.print_exc()
    card_inativo_html = "Erro ao processar dados de demissões"
    grafico_corrida_inativo = ""
    grafico_mes_inativo = ""
    grafico_tipo_inativo = ""
    tabela_inativos = ""

# ==============================================================================
# 4. ANÁLISE DE ADMISSÕES
# ==============================================================================

logging.info("="*80)
logging.info("4. ANÁLISE DE ADMISSÕES")
logging.info("="*80)

try:
    hc_admitidos = dados_hc.loc[dados_hc['ano_admissao'] == 2026].copy()
    logging.info(f"✓ {hc_admitidos.shape[0]} admissões em 2026 identificadas.")

    # Card headcount
    headcount_admitido = hc_admitidos.shape[0]
    fig_card_admitido = go.Figure(go.Indicator(
        mode="number",
        value=headcount_admitido,
        title={"text": "Admissões 2026"},
        number={'font': {'size': 60, 'color': '#28a745'}},
    ))
    fig_card_admitido.update_layout(height=150, margin=dict(l=0, r=0, t=30, b=0), paper_bgcolor="white")
    card_admitido_html = fig_card_admitido.to_html(full_html=False, include_plotlyjs='cdn')

    # Gráfico de Corrida
    if len(hc_admitidos) > 0:
        hc_admitidos['Data de Admissão'] = pd.to_datetime(hc_admitidos['data_admissao'], format='%d/%m/%Y', errors='coerce')
        hc_admitidos_corrida = hc_admitidos.dropna(subset=['Data de Admissão'])
        
        if len(hc_admitidos_corrida) > 0:
            contagem_corrida_adm = hc_admitidos_corrida.groupby(['empresa_resumo', 'Data de Admissão']).size().reset_index(name='Admitidos')
            
            todas_empresas_adm = contagem_corrida_adm['empresa_resumo'].unique()
            todas_datas_adm = pd.date_range(
                start=contagem_corrida_adm['Data de Admissão'].min(),
                end=contagem_corrida_adm['Data de Admissão'].max()
            )
            
            combinacoes_adm = pd.MultiIndex.from_product(
                [todas_empresas_adm, todas_datas_adm],
                names=['empresa_resumo', 'Data de Admissão']
            ).to_frame(index=False)
            
            contagem_completo_adm = pd.merge(combinacoes_adm, contagem_corrida_adm, on=['empresa_resumo', 'Data de Admissão'], how='left')
            contagem_completo_adm['Admitidos'] = contagem_completo_adm['Admitidos'].fillna(0)
            
            # 1. Ordenar e calcular o acumulado
            contagem_completo_adm = contagem_completo_adm.sort_values(by=['empresa_resumo', 'Data de Admissão'])
            contagem_completo_adm['Admissão Acumulado'] = contagem_completo_adm.groupby('empresa_resumo')['Admitidos'].cumsum()
            contagem_completo_adm = contagem_completo_adm.rename(columns={'empresa_resumo': 'Empresa'})

            # 2. Calcular o Rank (posição) de cada empresa a cada dia
            contagem_completo_adm['Rank'] = contagem_completo_adm.groupby('Data de Admissão')['Admissão Acumulado'].rank(method='first', ascending=True)

            # 3. Ordenar o dataframe pela Data e depois pelo Rank para a animação fluir corretamente
            contagem_completo_adm = contagem_completo_adm.sort_values(by=['Data de Admissão', 'Rank'])

            # Formatar a data para o padrão BR
            contagem_completo_adm['Data Formatada'] = contagem_completo_adm['Data de Admissão'].dt.strftime('%d/%m/%Y')
            
            # 4. Criar o texto combinando o nome da empresa e o valor para exibir na barra
            contagem_completo_adm['Texto_Barra'] = contagem_completo_adm['Empresa'] + ' - ' + contagem_completo_adm['Admissão Acumulado'].astype(int).astype(str)
            
            # Gerar o gráfico
            fig_corrida_admitido = px.bar(
                contagem_completo_adm,
                x='Admissão Acumulado',
                y='Rank', # Eixo Y dinâmico baseado no Rank
                orientation='h',
                color='Empresa',
                text='Texto_Barra', # Rótulo interno/externo com Empresa + Valor
                animation_frame='Data Formatada',
                animation_group='Empresa',
                title='Admissões Acumuladas por Unidade'
            )
            
            fig_corrida_admitido.update_layout(
                height=600,
                margin=dict(l=40, r=40, t=60, b=40),
                yaxis=dict(
                    showticklabels=False, # Oculta os números do eixo Y
                    title='',             # Remove o título do eixo Y
                    range=[0.5, len(todas_empresas_adm) + 0.5] # Trava a escala do eixo Y
                ),
                xaxis_title='Admissões Acumuladas'
            )
            
            # Posicionar o texto para melhor legibilidade
            fig_corrida_admitido.update_traces(textposition='outside')
            
            grafico_corrida_admitido = fig_corrida_admitido.to_html(full_html=False, include_plotlyjs=False)
        else:
            grafico_corrida_admitido = "<p>Sem dados para gráfico de corrida</p>"
    else:
        grafico_corrida_admitido = "<p>Sem dados de admissões em 2026</p>"

    # Gráfico por Mês
    if len(hc_admitidos) > 0:
        hc_admitidos['mes_admissao'] = pd.to_numeric(hc_admitidos['mes_admissao'], errors='coerce')
        meses_nome = {1: 'Jan', 2: 'Fev', 3: 'Mar', 4: 'Abr', 5: 'Mai', 6: 'Jun', 
                      7: 'Jul', 8: 'Ago', 9: 'Set', 10: 'Out', 11: 'Nov', 12: 'Dez'}
        
        contagem_mes_adm = hc_admitidos['mes_admissao'].value_counts().sort_index()
        
        if len(contagem_mes_adm) > 0:
            nomes_meses_adm = contagem_mes_adm.index.map(meses_nome)
            
    # --- AJUSTE PARA COMPARATIVO ANO CONTRA ANO (ADMISSÕES) ---
    df_yoy_adm = dados_hc.loc[dados_hc['ano_admissao'].isin([2025, 2026])].copy()
    
    # Converter ano para string
    df_yoy_adm['ano_admissao'] = df_yoy_adm['ano_admissao'].astype(int).astype(str)
    df_yoy_adm['mes_admissao'] = pd.to_numeric(df_yoy_adm['mes_admissao'], errors='coerce')

    contagem_yoy_adm = df_yoy_adm.groupby(['mes_admissao', 'ano_admissao']).size().reset_index(name='Quantidade')
    contagem_yoy_adm['Mês'] = contagem_yoy_adm['mes_admissao'].map(meses_nome)
    contagem_yoy_adm = contagem_yoy_adm.sort_values('mes_admissao')

    fig_mes_admitido = px.bar(
        contagem_yoy_adm,
        x='Mês',
        y='Quantidade',
        color='ano_admissao', 
        barmode='group',
        title='Admissões: 2025 vs 2026',
        color_discrete_map={'2025': '#05BEE8', '2026': '#3F62FC'} # Chaves como string
    )
    
    fig_mes_admitido.update_layout(
        height=400, 
        margin=dict(l=40, r=40, t=60, b=40), 
        legend_title_text='Ano',
        xaxis_title='Mês',
        yaxis_title='Quantidade')
    
    grafico_mes_admitido = fig_mes_admitido.to_html(full_html=False, include_plotlyjs=False)

    # Tabela de admitidos
    colunas_tabela_admitidos = ['registro', 'nome', 'idade', 'rg', 'cpf', 'data_admissao', 'situacao', 'data_rescisao', 
                                'dias_trabalhado', 'descricao_rescisao', 'cargo_completo', 'salario_total', 'secao',
                                'centro_custo', 'empresa_resumo', 'formacoes']

    tabela_admitidos = hc_admitidos[colunas_tabela_admitidos].to_html(index=False, classes='tabela-dados', border=0)

    logging.info("✓ SEÇÃO 4 CONCLUÍDA COM SUCESSO\n")

except Exception as e:
    logging.error(f"ERRO na SEÇÃO 4: {e}")
    traceback.print_exc()
    card_admitido_html = "Erro ao processar dados de admissões"
    grafico_corrida_admitido = "Erro ao gerar gráfico de corrida"
    grafico_mes_admitido = "Erro ao gerar gráfico por mês"
    tabela_admitidos = "Erro ao gerar tabela"

# ==============================================================================
# PREPARAÇÃO FINAL DOS DADOS
# ==============================================================================

logging.info("⚙️ Aplicando tipagem forte nas colunas antes da exportação...")

# Lista de DataFrames para processar
dfs_para_ajustar = [hc_ativos, hc_inativos, hc_admitidos]

for df in dfs_para_ajustar:
    # 1. Colunas de TEXTO
    cols_texto = ['registro', 'nome', 'rg', 'cpf']
    for col in cols_texto:
        if col in df.columns:
            # Converte para string e remove '.0' caso tenha vindo como float
            df[col] = df[col].astype(str).str.replace(r'\.0$', '', regex=True).replace('nan', '')

    # 2. Coluna de INTEIRO (dias_trabalhado)
    if 'dias_trabalhado' in df.columns:
        # Preenche vazios com 0, converte para inteiro
        df['dias_trabalhado'] = df['dias_trabalhado'].fillna(0).astype(int)

# ==============================================================================
# 5. GERAÇÃO DO RELATÓRIO HTML E EXCEL
# ==============================================================================

logging.info("="*80)
logging.info("5. GERAÇÃO DO RELATÓRIO HTML E EXCEL")
logging.info("="*80)

try:
    # Salvar arquivos Excel
    arquivo_ativos = 'HC_Ativos.xlsx'
    arquivo_demissoes = 'HC_Demissoes_2026.xlsx'
    arquivo_admissoes = 'HC_Admissoes_2026.xlsx'

    def converter_datas_para_string(df, colunas_data_str):
        df_copia = df.copy()
        
        for col in colunas_data_str:
            if col in df_copia.columns:
                if pd.api.types.is_datetime64_any_dtype(df_copia[col]):
                    df_copia[col] = df_copia[col].dt.strftime('%d/%m/%Y')
                df_copia[col] = df_copia[col].fillna('')
        
        return df_copia

    # Definir colunas de data
    # Força as colunas de data a irem como texto formatado para o arquivo Excel
    colunas_data = ['nascimento', 'data_admissao', 'data_rescisao', 'data_nasc_conjuge']
    for col in colunas_data:
        if col in hc_ativos.columns:
            # Se a coluna for do tipo datetime do pandas, transforma em string dd/mm/aaaa
            if pd.api.types.is_datetime64_any_dtype(hc_ativos[col]):
                hc_ativos[col] = hc_ativos[col].dt.strftime('%d/%m/%Y')
            else:
                hc_ativos[col] = hc_ativos[col].astype(str)
    
    logging.info("💾 Salvando e formatando Excel - Ativos...")
    hc_ativos_export = converter_datas_para_string(hc_ativos[colunas_tabela_ativos], colunas_data)
    hc_ativos_export.to_excel(arquivo_ativos, index=False, sheet_name='Ativos')
    formatar_excel_profissional(arquivo_ativos)
    
    logging.info("💾 Salvando e formatando Excel - Demissões...")
    hc_inativos_export = converter_datas_para_string(hc_inativos[colunas_tabela_inativos], colunas_data)
    hc_inativos_export.to_excel(arquivo_demissoes, index=False, sheet_name='Demissões 2026')
    formatar_excel_profissional(arquivo_demissoes)
    
    logging.info("💾 Salvando e formatando Excel - Admissões...")
    hc_admitidos_export = converter_datas_para_string(hc_admitidos[colunas_tabela_admitidos], colunas_data)
    hc_admitidos_export.to_excel(arquivo_admissoes, index=False, sheet_name='Admissões 2026')
    formatar_excel_profissional(arquivo_admissoes)

    logging.info("✓ Arquivos Excel salvos e formatados com sucesso\n")

except Exception as e:
    logging.error(f"ERRO ao salvar/formatar Excel na SEÇÃO 5: {e}")
    traceback.print_exc()

# ==============================================================================
# 6. ANÁLISE DE TURNOVER
# ==============================================================================
logging.info("Gerando análise de Turnover 2025 vs 2026...")

# 1. Converter as colunas corretas para datetime
dados_hc['data_rescisao'] = pd.to_datetime(dados_hc['data_rescisao'], dayfirst=True, errors='coerce')
dados_hc['data_admissao'] = pd.to_datetime(dados_hc['data_admissao'], dayfirst=True, errors='coerce')

# Imprimir no terminal as datas de demissão de 2026
demissoes_2026_check = dados_hc[dados_hc['data_rescisao'].dt.year == 2026]['data_rescisao']
logging.info(f"Datas de demissão identificadas em 2026:\n{demissoes_2026_check}")

# Gera lista de meses para análise
meses_analise = pd.date_range(start='2025-01-01', end='2026-12-31', freq='ME')
dados_turnover = []

for data_corte in meses_analise:
    ano = data_corte.year
    mes = data_corte.month
    
    # 2. Demissões no mês
    demissoes_mes = dados_hc[
        (dados_hc['data_rescisao'].dt.year == ano) & 
        (dados_hc['data_rescisao'].dt.month == mes)
    ].shape[0]
    
    # 3. Ativos no fim do mês
    ativos_fim_mes = dados_hc[
        (dados_hc['data_admissao'] <= data_corte) & 
        ((dados_hc['data_rescisao'].isna()) | (dados_hc['data_rescisao'] > data_corte))
    ].shape[0]
    
    # 4. Cálculo da Taxa %
    taxa = (demissoes_mes / ativos_fim_mes * 100) if ativos_fim_mes > 0 else 0
    
    dados_turnover.append({
        'Ano': str(ano),
        'Mes_Nome': data_corte.strftime('%b'),
        'Mes_Num': mes,
        'Taxa': taxa
    })

df_turnover = pd.DataFrame(dados_turnover)

# Criar Figura Plotly
fig_turnover = go.Figure()

# Linha 2025
df_2025 = df_turnover[df_turnover['Ano'] == '2025']
fig_turnover.add_trace(go.Scatter(
    x=df_2025['Mes_Nome'], y=df_2025['Taxa'],
    mode='lines+markers', name='2025',
    line=dict(color='#D97373', width=2)
))

# Linha 2026
df_2026 = df_turnover[df_turnover['Ano'] == '2026']
fig_turnover.add_trace(go.Scatter(
    x=df_2026['Mes_Nome'], y=df_2026['Taxa'],
    mode='lines+markers', name='2026',
    line=dict(color='#DEB521', width=4)
))

# Linha de Controle (Limite estipulado em 2%)
fig_turnover.add_hline(y=2, line_dash="dot", annotation_text="Limite 2%", line_color="red")

fig_turnover.update_layout(
    title='<b>Taxa de Turnover Mensal: 2025 vs 2026</b>',
    yaxis_title='Turnover (%)',
    template='plotly_white',
    height=400
)

# Converter para HTML string
grafico_turnover = fig_turnover.to_html(full_html=False, include_plotlyjs='cdn')

# ==============================================================================
# 7. ANÁLISE DE EARLY TURNOVER (< 90 DIAS)
# ==============================================================================
logging.info("Gerando análise de Early Turnover (Qualidade de Contratação)...")

# 1. Filtrar demissões de 2026
df_early = dados_hc[
    (dados_hc['data_rescisao'].dt.year == 2026)
].copy()

# 2. Calcular tempo de casa em dias
df_early['dias_trabalhados'] = (df_early['data_rescisao'] - df_early['data_admissao']).dt.days

# 3. Criar categoria (Early vs Orgânico)
df_early['tipo_saida'] = df_early['dias_trabalhados'].apply(
    lambda x: 'Early Turnover (<90 dias)' if x <= 90 else 'Turnover Orgânico (>90 dias)'
)

# --- CÁLCULOS MATEMÁTICOS ---
total_demissoes_2026 = df_early.shape[0]
early_count = df_early[df_early['dias_trabalhados'] <= 90].shape[0]

# Evitar divisão por zero
pct_early = (early_count / total_demissoes_2026 * 100) if total_demissoes_2026 > 0 else 0

# --- KPI GERAL (INDICADOR - GAUGE) ---
fig_early_gauge = go.Figure(go.Indicator(
    mode = "gauge+number",
    value = pct_early,
    number = {'suffix': "%", 'font': {'size': 40}}, 
    title = {'text': "<b>Early Turnover 2026 (Saídas < 90 dias)</b>"},
    gauge = {
        'axis': {'range': [None, 100]},
        'bar': {'color': "#ef4444"},
        'steps': [
            {'range': [0, 15], 'color': "#dcfce7"},
            {'range': [15, 30], 'color': "#fef9c3"},
            {'range': [30, 100], 'color': "#fee2e2"}
        ],
        'threshold': {
            'line': {'color': "red", 'width': 4},
            'thickness': 0.75,
            'value': 20
        }
    }
))

# Ajuste de Margens
fig_early_gauge.update_layout(
    height=350,
    margin=dict(l=30, r=30, t=80, b=30)
)

# --- GRÁFICO TEMPORAL (BARRAS EMPILHADAS) ---
# Agrupar por mês e tipo de saída
df_early['mes_num'] = df_early['data_rescisao'].dt.month
df_early['mes_nome'] = df_early['data_rescisao'].dt.strftime('%b')

resumo_early = df_early.groupby(['mes_num', 'mes_nome', 'tipo_saida']).size().reset_index(name='qtd')
resumo_early = resumo_early.sort_values('mes_num')

fig_early_bar = px.bar(
    resumo_early, 
    x='mes_nome', 
    y='qtd', 
    color='tipo_saida',
    title="<b>Perfil das Demissões: Recentes vs. Antigos (Mês a Mês)</b>",
    color_discrete_map={
        'Early Turnover (<90 dias)': '#ef4444',
        'Turnover Orgânico (>90 dias)': '#3b82f6'
    },
    text_auto=True
)

# Ajuste de Layout Vertical
fig_early_bar.update_layout(
    barmode='stack', 
    xaxis_title="Mês", 
    yaxis_title="Qtd. Demissões",
    legend_title="Tempo de Casa",
    template='plotly_white',
    height=400,
    margin=dict(t=50, b=50)
)

# Converter para HTML
grafico_early_gauge = fig_early_gauge.to_html(full_html=False, include_plotlyjs='cdn')
grafico_early_bar = fig_early_bar.to_html(full_html=False, include_plotlyjs='cdn')

# ==============================================================================
# 8. ANÁLISE DE SOBREVIVÊNCIA (TENURE / TEMPO DE CASA)
# ==============================================================================
logging.info("Gerando análise de Sobrevivência (Tenure dos Demitidos)...")

# 1. Filtrar demissões de 2026
df_tenure = dados_hc[
    (dados_hc['data_rescisao'].dt.year == 2026)
].copy()

# 2. Calcular tempo de casa em ANOS (para facilitar leitura)
df_tenure['anos_casa'] = (df_tenure['data_rescisao'] - df_tenure['data_admissao']).dt.days / 365.25

# --- GRÁFICO 1: HISTOGRAMA (DISTRIBUIÇÃO GERAL) ---
fig_tenure_hist = px.histogram(
    df_tenure, 
    x="anos_casa",
    nbins=20, # Ajuste a granularidade
    title="<b>Ponto de Ruptura: Com quanto tempo de casa as pessoas saem?</b>",
    labels={'anos_casa': 'Tempo de Casa (Anos)'},
    color_discrete_sequence=['#6366f1'],
    text_auto=True
)

fig_tenure_hist.update_layout(
    yaxis_title="Qtd. Demissões",
    xaxis_title="Anos de Empresa",
    bargap=0.1, # Espaço entre barras
    template='plotly_white',
    height=400
)

# Adiciona uma linha vertical na Média
media_anos = df_tenure['anos_casa'].mean()
fig_tenure_hist.add_vline(
    x=media_anos, 
    line_dash="dash", 
    line_color="red", 
    annotation_text=f"Média: {media_anos:.1f} anos", 
    annotation_position="top right"
)

# --- GRÁFICO 2: BOXPLOT POR CARGO (TOP 10 MAIS DEMITIDOS) ---
top_cargos_demissao = df_tenure['cargo_completo'].value_counts().nlargest(10).index
df_tenure_top = df_tenure[df_tenure['cargo_completo'].isin(top_cargos_demissao)]

fig_tenure_box = px.box(
    df_tenure_top, 
    x="anos_casa", 
    y="cargo_completo",
    title="<b>Variação do Tempo de Casa por Cargo (Top 10 Demissões)</b>",
    color="cargo_completo",
    points="all" # Mostra os pontos individuais
)

fig_tenure_box.update_layout(
    showlegend=False,
    xaxis_title="Anos de Empresa",
    yaxis_title="",
    template='plotly_white',
    height=500,
    margin=dict(l=10)
)

# Converter para HTML
grafico_tenure_hist = fig_tenure_hist.to_html(full_html=False, include_plotlyjs='cdn')
grafico_tenure_box = fig_tenure_box.to_html(full_html=False, include_plotlyjs='cdn')

# ==============================================================================
# 9. ANÁLISE FINANCEIRA (VISÃO FP&A - CUSTO DE SUBSTITUIÇÃO)
# ==============================================================================
logging.info("Gerando análise Financeira de Turnover...")

# 1. Configuração do Fator de Custo (Pode ajustar para 2.0 se quiser ser mais agressivo)
FATOR_CUSTO = 1.5 

# 2. Filtrar demissões de 2026
df_financeiro = dados_hc[
    (dados_hc['data_rescisao'].dt.year == 2026)
].copy()

# 3. Tratamento da Coluna de Salário (Garantir que é número)
col_salario = 'salario_atual' if 'salario_atual' in df_financeiro.columns else 'salario_total'

# Remove caracteres de moeda se for string e converte
if df_financeiro[col_salario].dtype == 'object':
    df_financeiro[col_salario] = df_financeiro[col_salario].astype(str).str.replace('R$', '', regex=False).str.replace('.', '', regex=False).str.replace(',', '.', regex=False)

df_financeiro[col_salario] = pd.to_numeric(df_financeiro[col_salario], errors='coerce').fillna(0)

# 4. Cálculos Financeiros
soma_salarios_demitidos = df_financeiro[col_salario].sum()
custo_total_estimado = soma_salarios_demitidos * FATOR_CUSTO
custo_medio_por_demissao = custo_total_estimado / len(df_financeiro) if len(df_financeiro) > 0 else 0

# --- VISUALIZAÇÃO (BIG NUMBERS) ---
fig_financeiro = go.Figure()

fig_financeiro.add_trace(go.Indicator(
    mode = "number",
    value = custo_total_estimado,
    number = {'prefix': "R$ ", 'valueformat': ",.2f", 'font': {'size': 50, 'color': '#b91c1c'}}, # Vermelho Escuro
    title = {
        'text': f"<b>Impacto Financeiro Estimado (YTD 2026)</b><br><span style='font-size:0.6em;color:gray'>Baseado em {FATOR_CUSTO}x o salário nominal (Rescisão + Recrutamento + Treinamento)</span>",
        'font': {'size': 18}
    },
    domain = {'x': [0, 1], 'y': [0, 1]}
))

fig_financeiro.update_layout(
    height=250,
    margin=dict(l=20, r=20, t=60, b=20),
    template='plotly_white'
)

# ==============================================================================
# 10. PAINEL DE KPIs ESTRATÉGICOS (CONSOLIDAÇÃO)
# ==============================================================================
logging.info("Gerando Painel de KPIs Estratégicos...")

# --- 1. CÁLCULO DO TURNOVER YTD (ACUMULADO 2026) ---
# Fórmula: Total Demissões 2026 / Média de Ativos 2026
total_demissoes_ytd = dados_hc[dados_hc['data_rescisao'].dt.year == 2026].shape[0]

# Média de ativos mensal em 2026
meses_2026 = pd.date_range(start='2026-01-01', end=pd.Timestamp.now(), freq='ME')
soma_ativos = 0
for data_corte in meses_2026:
    ativos = dados_hc[
        (dados_hc['data_admissao'] <= data_corte) & 
        ((dados_hc['data_rescisao'].isna()) | (dados_hc['data_rescisao'] > data_corte))
    ].shape[0]
    soma_ativos += ativos

media_ativos_ytd = soma_ativos / len(meses_2026) if len(meses_2026) > 0 else 0
taxa_turnover_ytd = (total_demissoes_ytd / media_ativos_ytd * 100) if media_ativos_ytd > 0 else 0

# --- 2. GERAÇÃO DOS CARDS ---
# HTML para ter flexibilidade total no layout
kpi_dashboard_html = f"""
<div style="display: flex; gap: 20px; margin-bottom: 30px; flex-wrap: wrap;">
    
    <!-- CARD 1: TURNOVER YTD -->
    <div style="flex: 1; min-width: 200px; background: white; padding: 20px; border-radius: 10px; box-shadow: 0 4px 6px rgba(0,0,0,0.1); border-left: 5px solid #3b82f6;">
        <h3 style="margin: 0; font-size: 14px; color: #64748b;">Turnover YTD (2026)</h3>
        <p style="margin: 10px 0 0 0; font-size: 28px; font-weight: bold; color: #1e293b;">{taxa_turnover_ytd:.1f}%</p>
        <span style="font-size: 12px; color: #94a3b8;">Acumulado do ano</span>
    </div>

    <!-- CARD 2: EARLY TURNOVER -->
    <div style="flex: 1; min-width: 200px; background: white; padding: 20px; border-radius: 10px; box-shadow: 0 4px 6px rgba(0,0,0,0.1); border-left: 5px solid #ef4444;">
        <h3 style="margin: 0; font-size: 14px; color: #64748b;">Early Turnover</h3>
        <p style="margin: 10px 0 0 0; font-size: 28px; font-weight: bold; color: #1e293b;">{pct_early:.1f}%</p>
        <span style="font-size: 12px; color: #94a3b8;">Saídas < 90 dias</span>
    </div>

    <!-- CARD 3: TEMPO MÉDIO (TENURE) -->
    <div style="flex: 1; min-width: 200px; background: white; padding: 20px; border-radius: 10px; box-shadow: 0 4px 6px rgba(0,0,0,0.1); border-left: 5px solid #8b5cf6;">
        <h3 style="margin: 0; font-size: 14px; color: #64748b;">Tempo Médio de Casa</h3>
        <p style="margin: 10px 0 0 0; font-size: 28px; font-weight: bold; color: #1e293b;">{media_anos:.1f} anos</p>
        <span style="font-size: 12px; color: #94a3b8;">Média dos demitidos</span>
    </div>

    <!-- CARD 4: CUSTO ESTIMADO -->
    <div style="flex: 1; min-width: 200px; background: white; padding: 20px; border-radius: 10px; box-shadow: 0 4px 6px rgba(0,0,0,0.1); border-left: 5px solid #10b981;">
        <h3 style="margin: 0; font-size: 14px; color: #64748b;">Custo Est. (YTD)</h3>
        <p style="margin: 10px 0 0 0; font-size: 28px; font-weight: bold; color: #1e293b;">R$ {custo_total_estimado:,.0f}</p>
        <span style="font-size: 12px; color: #94a3b8;">Impacto Financeiro</span>
    </div>

</div>
"""

# Converter para HTML
grafico_financeiro = fig_financeiro.to_html(full_html=False, include_plotlyjs='cdn')

    # HTML com referência aos arquivos Excel
try:
    html_final = f"""
<!DOCTYPE html>
<html lang="pt-BR">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Análise de HC - Indicadores de Gestão de Pessoas</title>
    <script src="https://cdn.plot.ly/plotly-2.18.2.min.js"></script>
    <!-- CRÍTICO: SheetJS importado ANTES do script customizado -->
    <script src="https://cdnjs.cloudflare.com/ajax/libs/xlsx/0.18.5/xlsx.full.min.js"></script>
    <style>
        * {{
            margin: 0;
            padding: 0;
            box-sizing: border-box;
        }}
        
        body {{
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            background-color: #f5f5f5;
            color: #333;
        }}
        
        .header {{
            background: linear-gradient(135deg, #0070C0 0%, #003d7a 100%);
            color: white;
            padding: 40px 20px;
            text-align: center;
            box-shadow: 0 2px 10px rgba(0,0,0,0.1);
        }}
        
        .header img {{
            max-width: 150px;
            margin-bottom: 20px;
        }}
        
        .header h1 {{
            font-size: 32px;
            margin-bottom: 10px;
        }}
        
        .container {{
            max-width: 1400px;
            margin: 0 auto;
            padding: 20px;
        }}
        
        .tabs {{
            display: flex;
            gap: 10px;
            margin-bottom: 30px;
            border-bottom: 2px solid #ddd;
            flex-wrap: wrap;
        }}
        
        .tab-button {{
            padding: 12px 24px;
            background-color: #f0f0f0;
            border: none;
            cursor: pointer;
            font-size: 16px;
            border-radius: 5px 5px 0 0;
            transition: all 0.3s;
            font-weight: 500;
        }}
        
        .tab-button.active {{
            background-color: #0070C0;
            color: white;
        }}
        
        .tab-button:hover {{
            background-color: #0070C0;
            color: white;
        }}
        
        .tab-content {{
            display: none;
            animation: fadeIn 0.3s;
        }}
        
        .tab-content.active {{
            display: block;
        }}
        
        @keyframes fadeIn {{
            from {{ opacity: 0; }}
            to {{ opacity: 1; }}
        }}
        
        .card {{
            background: white;
            border-radius: 8px;
            padding: 20px;
            margin-bottom: 20px;
            box-shadow: 0 2px 8px rgba(0,0,0,0.1);
        }}
        
        .card-header {{
            display: flex;
            justify-content: space-between;
            align-items: center;
            margin-bottom: 20px;
        }}
        
        .card-header h2 {{
            margin: 0;
            color: #0070C0;
            border: none;
            padding: 0;
        }}
        
        .download-button {{
            padding: 10px 20px;
            background-color: #28a745;
            color: white;
            border: none;
            cursor: pointer;
            font-size: 14px;
            border-radius: 5px;
            transition: all 0.3s;
            font-weight: 500;
            display: flex;
            align-items: center;
            gap: 8px;
        }}
        
        .download-button:hover {{
            background-color: #218838;
            transform: scale(1.05);
        }}
        
        .download-button:active {{
            transform: scale(0.98);
        }}
        
        /* ===== ESTILO DO BOTÃO VOLTAR AO TOPO ===== */
        .btn-voltar-topo {{
            position: fixed;
            bottom: 30px;
            right: 30px;
            width: 50px;
            height: 50px;
            background-color: #0070C0;
            color: white;
            border: none;
            border-radius: 50%;
            cursor: pointer;
            font-size: 24px;
            display: none;
            align-items: center;
            justify-content: center;
            box-shadow: 0 4px 12px rgba(0, 112, 192, 0.4);
            transition: all 0.3s ease;
            z-index: 1000;
        }}
        
        .btn-voltar-topo:hover {{
            background-color: #003d7a;
            transform: translateY(-3px);
            box-shadow: 0 6px 16px rgba(0, 112, 192, 0.6);
        }}
        
        .btn-voltar-topo:active {{
            transform: translateY(-1px);
        }}
        
        .btn-voltar-topo.visible {{
            display: flex;
        }}

        /* ===== FIM DO ESTILO DO BOTÃO ===== */
        .metrics {{
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(250px, 1fr));
            gap: 20px;
            margin-bottom: 30px;
        }}
        
        .grafico {{
            background: white;
            border-radius: 8px;
            padding: 20px;
            margin-bottom: 20px;
            box-shadow: 0 2px 8px rgba(0,0,0,0.1);
        }}
        
        .tabela-dados {{
            width: 100%;
            border-collapse: collapse;
            margin-top: 20px;
            font-size: 13px;
            max-height: 600px;
            overflow-y: auto;
            display: block;
        }}
        
        .tabela-dados thead {{
            display: table;
            width: 100%;
            background-color: #0070C0;
            color: white;
            position: sticky;
            top: 0;
        }}
        
        .tabela-dados tbody {{
            display: block;
            width: 100%;
            overflow-y: auto;
            max-height: 500px;
        }}
        
        .tabela-dados tr {{
            display: table;
            width: 100%;
            table-layout: fixed;
        }}
        
        .tabela-dados th {{
            background-color: #0070C0;
            color: white;
            padding: 10px;
            text-align: left;
            font-weight: 600;
        }}
        
        .tabela-dados td {{
            padding: 8px 10px;
            border-bottom: 1px solid #ddd;
        }}
        
        .tabela-dados tbody tr:hover {{
            background-color: #f9f9f9;
        }}
        
        .tabela-estatistica {{
            width: 100%;
            border-collapse: collapse;
            margin-top: 20px;
        }}
        
        .tabela-estatistica th {{
            background-color: #0070C0;
            color: white;
            padding: 10px;
            text-align: center;
            font-weight: 600;
        }}
        
        .tabela-estatistica td {{
            padding: 8px 10px;
            border: 1px solid #ddd;
            text-align: center;
        }}
        
        .footer {{
            text-align: center;
            padding: 20px;
            color: #666;
            border-top: 1px solid #ddd;
            margin-top: 40px;
        }}
        
        h2 {{
            color: #0070C0;
            margin-top: 30px;
            margin-bottom: 20px;
            border-bottom: 2px solid #0070C0;
            padding-bottom: 10px;
        }}
        
        .grid-2 {{
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(500px, 1fr));
            gap: 20px;
        }}
        
        @media (max-width: 768px) {{
            .card-header {{
                flex-direction: column;
                gap: 10px;
            }}
            
            .download-button {{
                width: 100%;
                justify-content: center;
            }}
            
            .btn-voltar-topo {{
                bottom: 20px;
                right: 20px;
                width: 45px;
                height: 45px;
                font-size: 20px;
            }}
        }}
    </style>
</head>
<body>
    <!-- ===== BOTÃO VOLTAR AO TOPO ===== -->
    <button class="btn-voltar-topo" id="btnVoltarTopo" title="Voltar ao topo">
        ↑
    </button>
    <!-- ===== FIM DO BOTÃO ===== -->
    
    <div class="header">
        <img src="{img_src_base64}" alt="Logo AFPESP">
        <h1>Análise de HC - Indicadores de Gestão de Pessoas</h1>
        <p>Relatório de Ativos, Demissões e Admissões</p>
    </div>
    
        <div class="container">
        <!-- DASHBOARD -->
        <h2>Resumo Estratégico 2026</h2>
        {kpi_dashboard_html}
    </div>
</body>

    <div class="container">
        <div class="tabs">
            <button class="tab-button active" onclick="abrirAba(event, 'ativos')">👥 Colaboradores Ativos</button>
            <button class="tab-button" onclick="abrirAba(event, 'demissoes')">📉 Demissões 2026</button>
            <button class="tab-button" onclick="abrirAba(event, 'admissoes')">📈 Admissões 2026</button>
        </div>
        
        <!-- ABA 1: ATIVOS -->
        <div id="ativos" class="tab-content active">
            <div class="metrics">
                {card_ativo_html}
            </div>
            
            <div class="card">
                <h2>Estatísticas Descritivas - Variáveis Quantitativas</h2>
                {tabela_estat_ativo}
            </div>
            
            <div class="grid-2">
                <div class="grafico">{grafico_sexo_ativo}</div>
                <div class="grafico">{grafico_filhos_ativo}</div>
            </div>
            
            <div class="grid-2">
                <div class="grafico">{grafico_etnia_ativo}</div>
                <div class="grafico">{grafico_civil_ativo}</div>
            </div>
            
            <div class="grafico">{grafico_form_ativo}</div>
            <div class="grafico">{grafico_cargo_ativo}</div>
            <div class="grafico">{grafico_cc_ativo}</div>
            <div class="grafico">{grafico_unidade_ativo}</div>
            
            <div class="grafico">{boxplot_ativo_html}</div>
            
            <!-- ===== SEÇÃO DE FILTROS - ATIVOS ===== -->
            <div class="card">
                <h2>🔍 Filtros de Pesquisa</h2>
                <div style="display: grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap: 15px; margin-bottom: 20px;">
                    
                    <!-- Filtro por Nome -->
                    <div>
                        <label for="filtroNome" style="font-weight: 600; display: block; margin-bottom: 5px;">Nome</label>
                        <input type="text" id="filtroNome" placeholder="Digite o nome..." 
                               style="width: 100%; padding: 8px; border: 1px solid #ddd; border-radius: 5px;">
                    </div>
                    
                    <!-- Filtro por Data de Admissão -->
                    <div>
                        <label for="filtroDataAdmissao" style="font-weight: 600; display: block; margin-bottom: 5px;">Data de Admissão (De)</label>
                        <input type="date" id="filtroDataAdmissao" 
                               style="width: 100%; padding: 8px; border: 1px solid #ddd; border-radius: 5px;">
                    </div>
                    
                    <!-- Filtro por Empresa -->
                    <div>
                        <label for="filtroEmpresa" style="font-weight: 600; display: block; margin-bottom: 5px;">Empresa</label>
                        <select id="filtroEmpresa" 
                                style="width: 100%; padding: 8px; border: 1px solid #ddd; border-radius: 5px;">
                            <option value="">-- Todas as Empresas --</option>
                        </select>
                    </div>
                    
                    <!-- Botão Limpar -->
                    <div style="display: flex; align-items: flex-end;">
                        <button onclick="limparFiltros()" 
                                style="width: 100%; padding: 8px 15px; background-color: #6c757d; color: white; border: none; border-radius: 5px; cursor: pointer; font-weight: 600;">
                            🔄 Limpar Filtros
                        </button>
                    </div>
                </div>
            </div>
            
            <!-- ===== TABELA COM FILTROS ===== -->
            <div class="card">
                <div class="card-header">
                    <h2>Tabela de Colaboradores Ativos</h2>
                    <div>
                        <button class="download-button" onclick="baixarExcel('{arquivo_ativos}')">
                            📥 Baixar Planilha Completa
                        </button>
                        <button class="download-button" style="background-color: #0070C0;
                        " onclick="exportarDadosFiltradosExcel('ativos', 'HC_Ativos_Filtrados.xlsx')">
                            ⬇️ Baixar Dados Filtrados
                        </button>
                    </div>
                </div>
                <div style="overflow-x: auto;">
                    {tabela_ativos}
                </div>
                <p id="totalRegistros" style="margin-top: 10px; color: #666; font-size: 14px;"></p>
            </div>
        </div>
        
        <!-- ABA 2: DEMISSÕES -->
        <div id="demissoes" class="tab-content">
            <div class="metrics">
                {card_inativo_html}
            </div>
            
            <div class="grafico">{grafico_corrida_inativo}</div>
            <div class="grafico">{grafico_mes_inativo}</div>
            <div class="card">
                <h2>Comparativo de Turnover (Rotatividade)</h2>
                <p>Acompanhamento da taxa mensal (Demissões / Ativos) independente do crescimento do HC.</p>
                {grafico_turnover}
            </div>
            <div class="grafico">{grafico_tipo_inativo}</div>
            
            <!-- Dentro da variável html_final -->
            <div class="card">
                <h2>Qualidade da Contratação (Early Turnover)</h2>
                <p>Monitoramento de desligamentos ocorridos durante o período de experiência (90 dias).</p>
                
                <!-- 1. Indicador Macro (Centralizado) -->
                <div style="width: 100%; max-width: 600px; margin: 0 auto;">
                    {grafico_early_gauge}
                </div>
                
                <hr style="border: 0; border-top: 1px solid #eee; margin: 20px 0;">

                <!-- 2. Gráfico Temporal (Largura Total) -->
                <div style="width: 100%;">
                    {grafico_early_bar}
                </div>
                
                <p style="font-size: 12px; color: gray; margin-top: 10px;">
                    * <b>Early Turnover (<90 dias):</b> Indica possíveis falhas no Recrutamento (perfil) ou Onboarding (integração).<br>
                    * <b>Turnover Orgânico (>90 dias):</b> Indica questões de Gestão, Clima, Remuneração ou Mercado.
                </p>
            </div>

                <div class="card">
                    <h2>Análise de Sobrevivência (Tenure)</h2>
                    <p>Identificação do "Ponto de Ruptura": Em qual momento da jornada o colaborador decide sair?</p>
                    
                    <!-- Gráfico de Distribuição -->
                    <div style="margin-bottom: 30px;">
                        {grafico_tenure_hist}
                        <p style="font-size: 12px; color: gray; text-align: center;">
                            * Barras altas indicam o período crítico onde ocorrem mais desligamentos.
                        </p>
                    </div>

                    <hr style="border: 0; border-top: 1px solid #eee; margin: 20px 0;">

                    <!-- Gráfico de Boxplot -->
                    <div>
                        {grafico_tenure_box}
                        <p style="font-size: 12px; color: gray;">
                            * <b>Interpretação do Boxplot:</b> A linha dentro da caixa é a Mediana. A caixa representa 50% das pessoas. 
                            Pontos fora das linhas são "outliers" (casos isolados).
                        </p>
                    </div>
                </div>

                    <div class="card" style="border-left: 5px solid #b91c1c;"> <!-- Borda vermelha para chamar atenção -->
                        <h2>Custo da Rotatividade</h2>
                        <p>Estimativa do impacto financeiro direto e indireto gerado pelas demissões no ano corrente.</p>
                        
                        {grafico_financeiro}
                        
                        <div style="background-color: #fef2f2; padding: 15px; border-radius: 8px; margin-top: 10px;">
                            <p style="margin: 0; color: #991b1b; font-size: 14px;">
                                <b>💡 O que compõe este custo (Fator 1.5x)?</b><br>
                                Não é apenas a rescisão. Inclui: Multa do FGTS, custos de recrutamento (anúncios, tempo de triagem), exames admissionais, 
                                treinamento do novo colaborador e a <i>curva de aprendizado</i> (tempo até o novo atingir a produtividade do anterior).
                            </p>
                        </div>
                    </div>

            <!-- ===== SEÇÃO DE FILTROS - DEMISSÕES ===== -->
            <div class="card">
                <h2>🔍 Filtros de Pesquisa</h2>
                <div style="display: grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap: 15px; margin-bottom: 20px;">
                    
                    <!-- Filtro por Nome -->
                    <div>
                        <label for="filtroNomeDemissoes" style="font-weight: 600; display: block; margin-bottom: 5px;">Nome</label>
                        <input type="text" id="filtroNomeDemissoes" placeholder="Digite o nome..." 
                               style="width: 100%; padding: 8px; border: 1px solid #ddd; border-radius: 5px;">
                    </div>
                    
                    <!-- Filtro por Data de Rescisão -->
                    <div>
                        <label for="filtroDataRescisao" style="font-weight: 600; display: block; margin-bottom: 5px;">Data de Rescisão (De)</label>
                        <input type="date" id="filtroDataRescisao" 
                               style="width: 100%; padding: 8px; border: 1px solid #ddd; border-radius: 5px;">
                    </div>
                    
                    <!-- Filtro por Empresa -->
                    <div>
                        <label for="filtroEmpresaDemissoes" style="font-weight: 600; display: block; margin-bottom: 5px;">Empresa</label>
                        <select id="filtroEmpresaDemissoes" 
                                style="width: 100%; padding: 8px; border: 1px solid #ddd; border-radius: 5px;">
                            <option value="">-- Todas as Empresas --</option>
                        </select>
                    </div>
                    
                    <!-- Filtro por Tipo de Rescisão -->
                    <div>
                        <label for="filtroTipoRescisao" style="font-weight: 600; display: block; margin-bottom: 5px;">Tipo de Rescisão</label>
                        <select id="filtroTipoRescisao" 
                                style="width: 100%; padding: 8px; border: 1px solid #ddd; border-radius: 5px;">
                            <option value="">-- Todos os Tipos --</option>
                        </select>
                    </div>
                    
                    <!-- Botão Limpar -->
                    <div style="display: flex; align-items: flex-end;">
                        <button onclick="limparFiltrosDemissoes()" 
                                style="width: 100%; padding: 8px 15px; background-color: #6c757d; color: white; border: none; border-radius: 5px; cursor: pointer; font-weight: 600;">
                            🔄 Limpar Filtros
                        </button>
                    </div>
                </div>
            </div>
            
            <div class="card">
                <div class="card-header">
                    <h2>Tabela de Demissões em 2026</h2>
                    <div>
                        <button class="download-button" onclick="baixarExcel('{arquivo_demissoes}')">
                            📥 Baixar Planilha Completa
                        </button>
                        <button class="download-button" style="background-color: #0070C0;" onclick="exportarDadosFiltradosExcel('demissoes', 'HC_Demissoes_Filtrados.xlsx')">
                            ⬇️ Baixar Dados Filtrados
                        </button>
                    </div>
                </div>
                <div style="overflow-x: auto;">
                    {tabela_inativos}
                </div>
                <p id="totalRegistrosDemissoes" style="margin-top: 10px; color: #666; font-size: 14px;"></p>
            </div>
        </div>
        
        <!-- ABA 3: ADMISSÕES -->
        <div id="admissoes" class="tab-content">
            <div class="metrics">
                {card_admitido_html}
            </div>
            
            <div class="grafico">{grafico_corrida_admitido}</div>
            <div class="grafico">{grafico_mes_admitido}</div>
            
            <!-- ===== SEÇÃO DE FILTROS - ADMISSÕES ===== -->
            <div class="card">
                <h2>🔍 Filtros de Pesquisa</h2>
                <div style="display: grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap: 15px; margin-bottom: 20px;">
                    
                    <!-- Filtro por Nome -->
                    <div>
                        <label for="filtroNomeAdmissoes" style="font-weight: 600; display: block; margin-bottom: 5px;">Nome</label>
                        <input type="text" id="filtroNomeAdmissoes" placeholder="Digite o nome..." 
                               style="width: 100%; padding: 8px; border: 1px solid #ddd; border-radius: 5px;">
                    </div>
                    
                    <!-- Filtro por Data de Admissão -->
                    <div>
                        <label for="filtroDataAdmissaoAdm" style="font-weight: 600; display: block; margin-bottom: 5px;">Data de Admissão (De)</label>
                        <input type="date" id="filtroDataAdmissaoAdm" 
                               style="width: 100%; padding: 8px; border: 1px solid #ddd; border-radius: 5px;">
                    </div>
                    
                    <!-- Filtro por Empresa -->
                    <div>
                        <label for="filtroEmpresaAdmissoes" style="font-weight: 600; display: block; margin-bottom: 5px;">Empresa</label>
                        <select id="filtroEmpresaAdmissoes" 
                                style="width: 100%; padding: 8px; border: 1px solid #ddd; border-radius: 5px;">
                            <option value="">-- Todas as Empresas --</option>
                        </select>
                    </div>
                    
                    <!-- Botão Limpar -->
                    <div style="display: flex; align-items: flex-end;">
                        <button onclick="limparFiltrosAdmissoes()" 
                                style="width: 100%; padding: 8px 15px; background-color: #6c757d; color: white; border: none; border-radius: 5px; cursor: pointer; font-weight: 600;">
                            🔄 Limpar Filtros
                        </button>
                    </div>
                </div>
            </div>
            
            <div class="card">
                <div class="card-header">
                    <h2>Tabela de Admissões em 2026</h2>
                    <div>
                        <button class="download-button" onclick="baixarExcel('{arquivo_admissoes}')">
                            📥 Baixar Planilha Completa
                        </button>
                        <button class="download-button" style="background-color: #0070C0;" onclick="exportarDadosFiltradosExcel('admissoes', 'HC_Admissoes_Filtrados.xlsx')">
                            ⬇️ Baixar Dados Filtrados
                        </button>
                    </div>
                </div>
                <div style="overflow-x: auto;">
                    {tabela_admitidos}
                </div>
                <p id="totalRegistrosAdmissoes" style="margin-top: 10px; color: #666; font-size: 14px;"></p>
            </div>
        </div>
    </div>
    
    <div class="footer">
        <p>Relatório gerado automaticamente em {datetime.now().strftime('%d/%m/%Y às %H:%M:%S')} | Gestão de Pessoas - AFPESP</p>
    </div>
    
<script>
        function abrirAba(evt, abaNome) {{
            var i, tabcontent, tabbuttons;
            tabcontent = document.getElementsByClassName("tab-content");
            for (i = 0; i < tabcontent.length; i++) {{
                tabcontent[i].classList.remove("active");
            }}
            tabbuttons = document.getElementsByClassName("tab-button");
            for (i = 0; i < tabbuttons.length; i++) {{
                tabbuttons[i].classList.remove("active");
            }}
            document.getElementById(abaNome).classList.add("active");
            if (evt && evt.currentTarget) {{
                evt.currentTarget.classList.add("active");
            }}
        }}

        // --- ENCONTRAR O ÍNDICE DA COLUNA PELO NOME DO CABEÇALHO ---
        function obterIndiceColuna(tabela, nomeCabecalho) {{
            var ths = Array.from(tabela.querySelectorAll('thead th'));
            return ths.findIndex(function(th) {{
                var texto = th.textContent.trim().toLowerCase();
                var busca = nomeCabecalho.toLowerCase();
                return texto.indexOf(busca) !== -1;
            }});
        }}

        // --- AUXILIAR PARA POPULAR OS SELECTS AUTOMATICAMENTE ---
function preencherSelectFiltroDinamico(tabelaId, nomeCabecalho, selectId) {{
            var aba = document.getElementById(tabelaId);
            if (!aba) return;
            var tabela = aba.querySelector('.tabela-dados');
            var select = document.getElementById(selectId);
            if (!tabela || !select) return;
            
            // Busca pelo termo exato ou abreviações comuns (Ex: 'desc' para descricao_rescisao)
            var idx = obterIndiceColuna(tabela, nomeCabecalho);
            if (idx === -1 && nomeCabecalho === 'Tipo Rescisão') {{
                idx = obterIndiceColuna(tabela, 'desc'); // tenta achar por descricao_rescisao
            }}
            if (idx === -1) return;
            
            var valores = new Set();
            var linhas = tabela.querySelectorAll('tbody tr');
            
            linhas.forEach(function(tr) {{
                if (tr.cells.length > idx) {{
                    var td = tr.cells[idx];
                    if (td) {{
                        var texto = td.textContent.trim();
                        if (texto) valores.add(texto);
                    }}
                }}
            }});
            
            while (select.options.length > 1) {{
                select.remove(1);
            }}
            
            Array.from(valores).sort().forEach(function(val) {{
                var opt = document.createElement('option');
                opt.value = val;
                opt.textContent = val;
                select.appendChild(opt);
            }});
        }}
        
        // --- ABA 1: ATIVOS ---
        function inicializarFiltros() {{
            preencherSelectFiltroDinamico('ativos', 'Empresa', 'filtroEmpresa');
            
            var fNome = document.getElementById('filtroNome');
            var fData = document.getElementById('filtroDataAdmissao');
            var fEmp = document.getElementById('filtroEmpresa');
            
            if (fNome) fNome.addEventListener('input', aplicarFiltros);
            if (fData) fData.addEventListener('change', aplicarFiltros);
            if (fEmp) fEmp.addEventListener('change', aplicarFiltros);
            
            aplicarFiltros();
        }}

function aplicarFiltros() {{
            var tabContent = document.getElementById('ativos');
            if (!tabContent) return;
            var tabela = tabContent.querySelector('.tabela-dados');
            if (!tabela) return;

            // Busca aos nomes reais das colunas geradas pelo Pandas
            var idxNome = obterIndiceColuna(tabela, 'nome');
            var idxEmpresa = obterIndiceColuna(tabela, 'empresa');
            var idxAdmissao = obterIndiceColuna(tabela, 'admiss'); // Procura por 'admissao' ou 'Admissão'

            var fNome = document.getElementById('filtroNome') ? document.getElementById('filtroNome').value.toLowerCase() : '';
            var fData = document.getElementById('filtroDataAdmissao') ? document.getElementById('filtroDataAdmissao').value : '';
            var fEmpresa = document.getElementById('filtroEmpresa') ? document.getElementById('filtroEmpresa').value : '';

            var linhas = tabela.querySelectorAll('tbody tr');
            var visiveisCount = 0;

            linhas.forEach(function(linha) {{
                var mostrar = true;

                if (fNome && idxNome !== -1) {{
                    var val = (linha.cells[idxNome] ? linha.cells[idxNome].textContent.toLowerCase() : '');
                    if (val.indexOf(fNome) === -1) mostrar = false;
                }}
                if (fEmpresa && idxEmpresa !== -1) {{
                    var val = (linha.cells[idxEmpresa] ? linha.cells[idxEmpresa].textContent.trim() : '');
                    if (val !== fEmpresa) mostrar = false;
                }}
                // Correção robusta da comparação de data
                if (fData && idxAdmissao !== -1) {{
                    var valStr = (linha.cells[idxAdmissao] ? linha.cells[idxAdmissao].textContent.trim() : '');
                    if (valStr) {{
                        var partes = valStr.split('/');
                        if (partes.length === 3) {{
                            // Formata no padrão numérico AAAAMMDD para comparação direta
                            var dataLinhaNum = partes[2] + partes[1].padStart(2, '0') + partes[0].padStart(2, '0');
                            var dataFiltroNum = fData.replace(/-/g, '');
                            
                            if (dataLinhaNum < dataFiltroNum) mostrar = false;
                        }} else {{
                            mostrar = false;
                        }}
                    }} else {{
                        mostrar = false;
                    }}
                }}

                linha.style.display = mostrar ? 'table-row' : 'none';
                if (mostrar) visiveisCount++;
            }});
            
            var totalReg = document.getElementById('totalRegistros');
            if (totalReg) {{
                totalReg.textContent = " 📊  Exibindo " + visiveisCount + " de " + linhas.length + " registros";
            }}
        }}

        function limparFiltros() {{
            if (document.getElementById('filtroNome')) document.getElementById('filtroNome').value = '';
            if (document.getElementById('filtroDataAdmissao')) document.getElementById('filtroDataAdmissao').value = '';
            if (document.getElementById('filtroEmpresa')) document.getElementById('filtroEmpresa').value = '';
            aplicarFiltros();
        }}

        // --- ABA 2: DEMISSÕES ---
        function inicializarFiltrosDemissoes() {{
            preencherSelectFiltroDinamico('demissoes', 'Empresa', 'filtroEmpresaDemissoes');
            preencherSelectFiltroDinamico('demissoes', 'Tipo Rescisão', 'filtroTipoRescisao');
            
            var fNome = document.getElementById('filtroNomeDemissoes');
            var fData = document.getElementById('filtroDataRescisao');
            var fEmp = document.getElementById('filtroEmpresaDemissoes');
            var fTipo = document.getElementById('filtroTipoRescisao');
            
            if (fNome) fNome.addEventListener('input', aplicarFiltrosDemissoes);
            if (fData) fData.addEventListener('change', aplicarFiltrosDemissoes);
            if (fEmp) fEmp.addEventListener('change', aplicarFiltrosDemissoes);
            if (fTipo) fTipo.addEventListener('change', aplicarFiltrosDemissoes);
            
            aplicarFiltrosDemissoes();
        }}

function aplicarFiltrosDemissoes() {{
    var tabContent = document.getElementById('demissoes');
    if (!tabContent) return;
    var tabela = tabContent.querySelector('.tabela-dados');
    if (!tabela) return;

    // BUSCA CIRÚRGICA E EXATA DOS CABEÇALHOS DO PANDAS
    var idxNome = obterIndiceColuna(tabela, 'nome');
    var idxEmpresa = obterIndiceColuna(tabela, 'empresa');
    
    // Forçamos termos longos para não confundir 'Data Admissão' com 'Data Rescisão'
    var idxRescisao = obterIndiceColuna(tabela, 'data_rescisao');
    if (idxRescisao === -1) idxRescisao = obterIndiceColuna(tabela, 'data rescisã');
    if (idxRescisao === -1) idxRescisao = obterIndiceColuna(tabela, 'rescisão'); // fallback seguro

    var idxTipoResc = obterIndiceColuna(tabela, 'descricao_rescisao');
    if (idxTipoResc === -1) idxTipoResc = obterIndiceColuna(tabela, 'descrição rescisã');
    if (idxTipoResc === -1) idxTipoResc = obterIndiceColuna(tabela, 'tipo rescisã');

    var fNome = document.getElementById('filtroNomeDemissoes') ? document.getElementById('filtroNomeDemissoes').value.toLowerCase() : '';
    var fData = document.getElementById('filtroDataRescisao') ? document.getElementById('filtroDataRescisao').value : '';
    var fEmpresa = document.getElementById('filtroEmpresaDemissoes') ? document.getElementById('filtroEmpresaDemissoes').value : '';
    var fTipo = document.getElementById('filtroTipoRescisao') ? document.getElementById('filtroTipoRescisao').value : '';

    var linhas = tabela.querySelectorAll('tbody tr');
    var visiveisCount = 0;

    linhas.forEach(function(linha) {{
        var mostrar = true;

        if (fNome && idxNome !== -1) {{
            var val = (linha.cells[idxNome] ? linha.cells[idxNome].textContent.toLowerCase() : '');
            if (val.indexOf(fNome) === -1) mostrar = false;
        }}
        if (fEmpresa && idxEmpresa !== -1) {{
            var val = (linha.cells[idxEmpresa] ? linha.cells[idxEmpresa].textContent.trim() : '');
            if (val !== fEmpresa) mostrar = false;
        }}
        if (fTipo && idxTipoResc !== -1) {{
            var val = (linha.cells[idxTipoResc] ? linha.cells[idxTipoResc].textContent.trim() : '');
            if (val !== fTipo) mostrar = false;
        }}
        
        // COMPARAÇÃO NUMÉRICA DO DIA REAL DA DEMISSÃO
        if (fData && idxRescisao !== -1) {{
            var valStr = (linha.cells[idxRescisao] ? linha.cells[idxRescisao].textContent.trim() : '');
            if (valStr) {{
                var partes = valStr.split('/');
                if (partes.length === 3) {{
                    // Formata em AAAAMMDD para comparar matematicamente sem falhas
                    var dataLinhaNum = partes[2] + partes[1].padStart(2, '0') + partes[0].padStart(2, '0');
                    var dataFiltroNum = fData.replace(/-/g, '');
                    
                    if (dataLinhaNum < dataFiltroNum) mostrar = false;
                }} else {{
                    mostrar = false;
                }}
            }} else {{
                mostrar = false; 
            }}
        }}

        linha.style.display = mostrar ? 'table-row' : 'none';
        if (mostrar) visiveisCount++;
    }});
    
    var txtContador = document.getElementById('totalRegistrosDemissoes');
    if (txtContador) {{
        txtContador.textContent = " 📊  Exibindo " + visiveisCount + " de " + linhas.length + " registros";
    }}
}}

        function limparFiltrosDemissoes() {{
            if (document.getElementById('filtroNomeDemissoes')) document.getElementById('filtroNomeDemissoes').value = '';
            if (document.getElementById('filtroDataRescisao')) document.getElementById('filtroDataRescisao').value = '';
            if (document.getElementById('filtroEmpresaDemissoes')) document.getElementById('filtroEmpresaDemissoes').value = '';
            if (document.getElementById('filtroTipoRescisao')) document.getElementById('filtroTipoRescisao').value = '';
            aplicarFiltrosDemissoes();
        }}

        // --- ABA 3: ADMISSÕES ---
        function inicializarFiltrosAdmissoes() {{
            preencherSelectFiltroDinamico('admissoes', 'Empresa', 'filtroEmpresaAdmissoes');
            
            var fNome = document.getElementById('filtroNomeAdmissoes');
            var fData = document.getElementById('filtroDataAdmissaoAdm');
            var fEmp = document.getElementById('filtroEmpresaAdmissoes');
            
            if (fNome) fNome.addEventListener('input', aplicarFiltrosAdmissoes);
            if (fData) fData.addEventListener('change', aplicarFiltrosAdmissoes);
            if (fEmp) fEmp.addEventListener('change', aplicarFiltrosAdmissoes);
            
            aplicarFiltrosAdmissoes();
        }}

function aplicarFiltrosAdmissoes() {{
            var tabContent = document.getElementById('admissoes');
            if (!tabContent) return;
            var tabela = tabContent.querySelector('.tabela-dados');
            if (!tabela) return;

            var idxNome = obterIndiceColuna(tabela, 'nome');
            var idxEmpresa = obterIndiceColuna(tabela, 'empresa');
            var idxAdmissao = obterIndiceColuna(tabela, 'admiss'); // Busca tolerante (pega 'data_admissao' ou 'Admissão')

            var fNome = document.getElementById('filtroNomeAdmissoes') ? document.getElementById('filtroNomeAdmissoes').value.toLowerCase() : '';
            // ALINHAMENTO CRÍTICO: Capturando o ID correto 'filtroDataAdmissaoAdm' usado no HTML dessa aba
            var fData = document.getElementById('filtroDataAdmissaoAdm') ? document.getElementById('filtroDataAdmissaoAdm').value : '';
            var fEmpresa = document.getElementById('filtroEmpresaAdmissoes') ? document.getElementById('filtroEmpresaAdmissoes').value : '';

            var linhas = tabela.querySelectorAll('tbody tr');
            var visiveisCount = 0;

            linhas.forEach(function(linha) {{
                var mostrar = true;

                if (fNome && idxNome !== -1) {{
                    var val = (linha.cells[idxNome] ? linha.cells[idxNome].textContent.toLowerCase() : '');
                    if (val.indexOf(fNome) === -1) mostrar = false;
                }}
                if (fEmpresa && idxEmpresa !== -1) {{
                    var val = (linha.cells[idxEmpresa] ? linha.cells[idxEmpresa].textContent.trim() : '');
                    if (val !== fEmpresa) mostrar = false;
                }}
                
                // COMPARAÇÃO NUMÉRICA BRUTA (EVITA ERRO DE RETORNO VAZIO)
                if (fData && idxAdmissao !== -1) {{
                    var valStr = (linha.cells[idxAdmissao] ? linha.cells[idxAdmissao].textContent.trim() : '');
                    if (valStr) {{
                        var partes = valStr.split('/');
                        if (partes.length === 3) {{
                            var dataLinhaNum = partes[2] + partes[1].padStart(2, '0') + partes[0].padStart(2, '0');
                            var dataFiltroNum = fData.replace(/-/g, '');
                            
                            // Mostra apenas registros a partir daquela data (Maior ou Igual)
                            if (dataLinhaNum < dataFiltroNum) mostrar = false;
                        }} else {{
                            mostrar = false;
                        }}
                    }} else {{
                        mostrar = false;
                    }}
                }}

                linha.style.display = mostrar ? 'table-row' : 'none';
                if (mostrar) visiveisCount++;
            }});
            
            var txtContador = document.getElementById('totalRegistrosAdmissoes');
            if (txtContador) {{
                txtContador.textContent = " 📊  Exibindo " + visiveisCount + " de " + linhas.length + " registros";
            }}
        }}

        function limparFiltrosAdmissoes() {{
            if (document.getElementById('filtroNomeAdmissoes')) document.getElementById('filtroNomeAdmissoes').value = '';
            if (document.getElementById('filtroDataAdmissaoAdm')) document.getElementById('filtroDataAdmissaoAdm').value = '';
            if (document.getElementById('filtroEmpresaAdmissoes')) document.getElementById('filtroEmpresaAdmissoes').value = '';
            aplicarFiltrosAdmissoes();
        }}

        // --- EXPORTAÇÃO EXCEL ---
        function exportarDadosFiltradosExcel(tabId, filename) {{
            let retryCount = 0;
            const maxRetries = 10;
            const retryDelay = 200;
            
            function doExport() {{
                if (typeof XLSX === 'undefined') {{
                    if (retryCount < maxRetries) {{
                        console.warn("XLSX is not defined yet. Retrying...");
                        retryCount++;
                        setTimeout(doExport, retryDelay);
                    }} else {{
                        alert('Erro ao carregar biblioteca Excel.');
                    }}
                    return;
                }}
                try {{
                    var tabContent = document.getElementById(tabId);
                    var tabela = tabContent.querySelector('.tabela-dados');
                    if (!tabela) return;

                    var headers = Array.from(tabela.querySelectorAll('thead th')).map(function(th) {{ return th.textContent.trim(); }});
                    var linhasVisiveis = Array.from(tabela.querySelectorAll('tbody tr')).filter(function(tr) {{ return tr.style.display !== 'none'; }});

                    if (linhasVisiveis.length === 0) {{
                        alert('Nenhum registro visível para exportar.');
                        return;
                    }}

                    var dataRows = [];
                    linhasVisiveis.forEach(function(linha) {{
                        var cells = Array.from(linha.querySelectorAll('td'));
                        var row = [];
                        cells.forEach(function(td) {{
                            row.push(td.textContent.trim());
                        }});
                        dataRows.push(row);
                    }});

                    var wb = XLSX.utils.book_new();
                    var ws = XLSX.utils.aoa_to_sheet([headers].concat(dataRows));
                    
                    var wscols = headers.map(function(h, i) {{
                        var max_len = h.length;
                        dataRows.forEach(function(row) {{
                            if (row[i] && row[i].length > max_len) max_len = row[i].length;
                        }});
                        return {{ wch: Math.min(max_len + 3, 50) }};
                    }});
                    ws['!cols'] = wscols;

                    XLSX.utils.book_append_sheet(wb, ws, "Dados Filtrados");
                    XLSX.writeFile(wb, filename);
                }} catch (err) {{
                    console.error('Erro na exportação Excel:', err);
                }}
            }}
            doExport();
        }}

        // --- SISTEMA: VOLTAR AO TOPO ---
        window.addEventListener('scroll', function() {{
            var btn = document.getElementById('btnVoltarTopo');
            if (btn) {{
                if (window.scrollY > 300) {{
                    btn.classList.add('visible');
                }} else {{
                    btn.classList.remove('visible');
                }}
            }}
        }});

        window.addEventListener('DOMContentLoaded', function() {{
            var btnTopo = document.getElementById('btnVoltarTopo');
            if (btnTopo) {{
                btnTopo.addEventListener('click', function() {{
                    window.scrollTo({{ top: 0, behavior: 'smooth' }});
                }});
            }}
        }});

        window.addEventListener('load', function() {{
            inicializarFiltros();
            inicializarFiltrosDemissoes();
            inicializarFiltrosAdmissoes();
        }});
    </script>
</body>
</html>
"""

    # Salvar HTML
    caminho_html = 'Relatório_HC.html'
    with open(caminho_html, 'w', encoding='utf-8') as f:
        f.write(html_final)

    logging.info(f"✓ Relatório HTML gerado: {caminho_html}")

    # Abrir no navegador
    webbrowser.open('file://' + os.path.abspath(caminho_html))
    logging.info("✓ Relatório aberto no navegador.\n")

except Exception as e:
    logging.error(f"ERRO ao gerar HTML na SEÇÃO 5: {e}")
    traceback.print_exc()
    sys.exit("Script encerrado.")

logging.info("="*80)
logging.info("✓ SCRIPT CONCLUÍDO COM SUCESSO!")
logging.info("="*80)

[2026-06-26 10:14:12] [INFO] ================================================================================
[2026-06-26 10:14:12] [INFO] 1. CARREGAMENTO E PRÉ-PROCESSAMENTO DOS DADOS
[2026-06-26 10:14:12] [INFO] ================================================================================


✓ Bibliotecas importadas com sucesso.



[2026-06-26 10:14:22] [INFO] ✓ Base de dados carregada com 9811 registros e 80 colunas.
[2026-06-26 10:14:22] [INFO] ✓ Colunas de data convertidas.
[2026-06-26 10:14:22] [INFO] ✓ SEÇÃO 1 CONCLUÍDA COM SUCESSO

[2026-06-26 10:14:22] [INFO] ================================================================================
[2026-06-26 10:14:22] [INFO] 2. ANÁLISE DE COLABORADORES ATIVOS
[2026-06-26 10:14:22] [INFO] ================================================================================
[2026-06-26 10:14:22] [INFO] ✓ 1764 colaboradores ativos identificados.
[2026-06-26 10:14:23] [INFO] ✓ SEÇÃO 2 CONCLUÍDA COM SUCESSO

[2026-06-26 10:14:23] [INFO] ================================================================================
[2026-06-26 10:14:23] [INFO] 3. ANÁLISE DE DEMISSÕES
[2026-06-26 10:14:23] [INFO] ================================================================================
[2026-06-26 10:14:23] [INFO] ✓ 168 demissões em 2026 identificadas.
[2026-06-26 10:14:44] [INFO] ✓ 

# Encaminhando E-mail

In [2]:
# ---- SECTION 1: CustomHandler (serve files as attachments) ----
class CustomHandler(SimpleHTTPRequestHandler):
    def end_headers(self):
        if self.path.endswith('.html'):
            self.send_header('Content-Disposition', 'attachment')
        super().end_headers()

# ---- SECTION 2: Start file server ----
def iniciar_servidor_arquivos():
    pasta_base = r'X:\Gestão de Pessoas\Analytics\08 - Notebooks Python\08.5 - Estudos e Projetos\Análise de Headcount'
    port = 8888
    server = HTTPServer(('', port), CustomHandler)
    thread = threading.Thread(target=server.serve_forever, daemon=True)
    thread.start()
    logging.info(f'Servidor de arquivos iniciado na porta {port}, servindo pasta: {pasta_base}')

# ---- SECTION 3: Load .env ----
def carregar_env():
    try:
        script_dir = os.path.dirname(os.path.abspath(__file__))
        env_path = os.path.join(script_dir, '.env')
    except NameError:
        script_dir = r'X:\Gestão de Pessoas\Analytics\08 - Notebooks Python\08.5 - Estudos e Projetos\Análise de Headcount'
        env_path = os.path.join(script_dir, '.env')
    
    if os.path.isfile(env_path):
        load_dotenv(env_path)
        logging.info(f'Arquivo .env carregado de: {env_path}')
    else:
        logging.warning(f'Arquivo .env não encontrado em: {env_path}')

# ---- SECTION 4: Validate Windows ----
def validar_windows():
    if sys.platform != 'win32':
        raise SystemError('Este script só pode ser executado no Windows (Outlook requer win32).')
    logging.info('Sistema Windows validado.')

# ---- SECTION 5: Connect to Outlook ----
def conectar_outlook():
    pythoncom.CoInitialize()
    outlook = win32.Dispatch('Outlook.Application')
    logging.info('Conectado ao Outlook.')
    return outlook

# ---- SECTION 6: Extract metrics from HTML ----
import re
import json

# Cálculo do Turnover Acumulado de 2026
turnover_acumulado = f"{taxa_turnover_ytd:.1f}%"

# Early Turnover
early_turnover = f"{pct_early:.1f}%"

def extrair_metricas_do_html(caminho_arquivo):
    with open(caminho_arquivo, 'r', encoding='utf-8') as f:
        html = f.read()

    padroes = [
        (r'"title":\{"text":"Headcount Ativo"\}[^}]*"value":(\d+)', 'headcount_ativo'),
        (r'"title":\{"text":"Demissões[^"]*"\}[^}]*"value":(\d+)', 'headcount_inativo'),
        (r'"title":\{"text":"Admissões[^"]*"\}[^}]*"value":(\d+)', 'headcount_admitido'),
        (r'"title":\{"text":"Turnover Acumulado 2026[^"]*"\}[^}]*"value":(\d+)', 'turnover_acumulado')
    ]

    resultados = {}
    for padrao, nome in padroes:
        match = re.search(padrao, html)
        if match:
            valor = int(match.group(1))
            resultados[nome] = valor
            print(f"Padrão '{padrao}': encontrado valor {valor}")
        else:
            resultados[nome] = 0
            print(f"Padrão '{padrao}': não encontrado")
    
    return resultados

now = datetime.now().strftime('%d/%m/%Y %H:%M:%S')

# ---- SECTION 7: Build email ----
def montar_email(outlook, pasta_base, destinatarios, assunto):
    caminho_html = os.path.join(pasta_base, 'Relatório_HC.html')
    metricas = extrair_metricas_do_html(caminho_html)
    corpo_html = f'''
    <html>
    <body>
        <p>Prezados,</p>
        <p>Segue abaixo as métricas extraídas do relatório:</p>
        <ul>
            <li><strong>Headcount Ativo:</strong> {metricas['headcount_ativo']}</li>
            <li><strong>Demissões (Inativos):</strong> {metricas['headcount_inativo']}</li>
            <li><strong>Admissões:</strong> {metricas['headcount_admitido']}</li>
            <li><strong>Turnover Acumulado 2026:</strong> {turnover_acumulado}</li>
            <li><strong>Demissões < 90 dias:</strong> {early_turnover}</li>              
        </ul>
        <p>Em anexo, o relatório completo em HTML gerado em {now}.</p>
        <p>Atenciosamente,<br><em><strong>Rodrigo Bernardes - People Analytics<br>Gestão de Pessoas<em><strong></p>
    </body>
    </html>
    '''
    mail = outlook.CreateItem(0)
    #mail.To = destinatarios (e-mails ficam no campo de cópia, visível a todos)
    mail.BCC= destinatarios
    mail.Subject = assunto
    mail.HTMLBody = corpo_html
    mail.Attachments.Add(caminho_html)
    logging.info('E-mail montado com sucesso.')
    return mail

# ---- SECTION 8: Send email ----
def enviar_email(mail):
    mail.Send()
    logging.info('E-mail enviado com sucesso.')

# ---- SECTION 9: Orchestrator ----
def orquestrador():
    # Passo 1: Carregar variáveis de ambiente
    carregar_env()
    
    # Passo 2: Validar sistema Windows
    validar_windows()
    
    # Passo 3: Iniciar servidor de arquivos
    iniciar_servidor_arquivos()
    
    # Passo 4: Conectar ao Outlook
    outlook = conectar_outlook()
    
    # Passo 5: Montar e-mail
    pasta_base = os.getenv('PASTA_BASE', r'X:\Gestão de Pessoas\Analytics\08 - Notebooks Python\08.5 - Estudos e Projetos\Análise de Headcount')
    destinatarios = os.getenv('EMAIL_DESTINATARIOS')
    assunto = os.getenv('EMAIL_ASSUNTO', 'Relatório de Headcount - Atualização Automática')
    if not destinatarios:
        logging.error('Variável EMAIL_DESTINATARIOS não definida. Encerrando.')
        return
    mail = montar_email(outlook, pasta_base, destinatarios, assunto)
    
    # Passo 6: Enviar e-mail
    enviar_email(mail)

# ---- SECTION 10: Main ----
if __name__ == '__main__':
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(levelname)s - %(message)s',
        handlers=[
            logging.StreamHandler(sys.stdout)
        ]
    )
    orquestrador()

tempo_1 = [id, datetime.today(), 1]

print('----------------------------------------------------------------------------------------------------')
print('')
print('     ✅  Processo finalizado')
print('')
print('     ⏱️   Tempo de execução:')
print('')
print(f'   {tempo_1[1] - tempo_0[1]}')
print('')
print('----------------------------------------------------------------------------------------------------')

[2026-06-26 10:15:04] [INFO] Arquivo .env carregado de: X:\Gestão de Pessoas\Analytics\08 - Notebooks Python\08.5 - Estudos e Projetos\Análise de Headcount\.env
[2026-06-26 10:15:04] [INFO] Sistema Windows validado.
[2026-06-26 10:15:04] [INFO] Servidor de arquivos iniciado na porta 8888, servindo pasta: X:\Gestão de Pessoas\Analytics\08 - Notebooks Python\08.5 - Estudos e Projetos\Análise de Headcount
[2026-06-26 10:15:05] [INFO] Conectado ao Outlook.


Padrão '"title":\{"text":"Headcount Ativo"\}[^}]*"value":(\d+)': encontrado valor 1764
Padrão '"title":\{"text":"Demissões[^"]*"\}[^}]*"value":(\d+)': encontrado valor 168
Padrão '"title":\{"text":"Admissões[^"]*"\}[^}]*"value":(\d+)': encontrado valor 159
Padrão '"title":\{"text":"Turnover Acumulado 2026[^"]*"\}[^}]*"value":(\d+)': não encontrado


[2026-06-26 10:15:05] [INFO] E-mail montado com sucesso.
[2026-06-26 10:15:05] [INFO] E-mail enviado com sucesso.


----------------------------------------------------------------------------------------------------

     ✅  Processo finalizado

     ⏱️   Tempo de execução:

   0:00:53.581369

----------------------------------------------------------------------------------------------------
